# Phần 3 — Lập lịch phân công

Hai bài lớn nhất của báo cáo, và là nơi hai luận điểm phương pháp lộ rõ nhất.

**Bài 3.1** là chỗ gặp cái bẫy nguy hiểm nhất trong cả dự án: hai mẫu chính thức
của IBM cùng mang tên *sports scheduling* nhưng là **hai bài toán khác nhau** —
bản OPL là double round-robin có sân nhà/khách và tối thiểu số break, bản docplex
là lịch NFL hai bảng đấu, **không có khái niệm home/away nên không có break**.
Xếp số liệu hai bản này cạnh nhau là so hai thứ khác nhau mà nhìn bảng không thấy
được. Cách xử lý: giữ nguyên cả hai mẫu chính thức, và viết thêm một bản port
biến thể A sang `docplex.cp` để hai trục so sánh có số liệu **trên cùng một bài**.

**Bài 3.2** là bài duy nhất có benchmark định lượng, và là bài chứng minh vì sao
báo cáo cần đủ ba chiều: so thẳng OPL với OR-Tools ở đó cho kết luận **ngược** với
sự thật, cho tới khi thêm điểm đo thứ ba.

Cả hai bài đều chạm trần Community Edition theo hai kiểu khác nhau — chi tiết ở
mục (d) của từng bài.

In [1]:
import sys; sys.path.insert(0, "../tools")
from nbutil import *

---

# Bài 3.1 — Lập lịch thi đấu thể thao (double round-robin)

## (a) Phát biểu bài toán

In [2]:
show_notes("3.1_sports", "Phát biểu")

## (a) Phát biểu bài toán

Một giải đấu có $n$ đội ($n$ chẵn). Mỗi đội có một **sân nhà**.

**Thể thức double round-robin.** Mỗi cặp đội gặp nhau đúng **hai lần** trong mùa
giải: một lần trên sân của đội này, một lần trên sân của đội kia. Trong mỗi trận,
đội đá trên sân của mình gọi là **home team**, đội còn lại là **away team**. Tổng
cộng có $n(n-1)$ trận — mỗi cặp có thứ tự (nhà, khách) đúng một trận.

**Lịch.** Mùa giải dài $W = 2(n-1)$ tuần. Mỗi tuần có đúng $n/2$ chỗ đá giống hệt
nhau, và **mỗi đội đá đúng một trận mỗi tuần**. Nhân lên: $2(n-1)\cdot\frac n2 = n(n-1)$
chỗ đá cho $n(n-1)$ trận — vừa khít, không thừa không thiếu chỗ nào.

**Ràng buộc thi đấu.**

1. **Hai nửa mùa giải.** Hai lượt của cùng một cặp đội phải rơi vào hai nửa khác
   nhau của mùa giải (nửa đầu: tuần $1..W/2$; nửa sau: tuần $W/2+1..W$).
2. **Giãn cách hai lượt.** Ngoài ra hai lượt đó phải cách nhau ít nhất
   $\delta=\min(n/2,\,6)$ tuần.
3. **Mở màn và khép lại.** Mỗi đội phải đá sân nhà ở tuần đầu **hoặc** tuần cuối,
   nhưng **không cả hai**.
4. **Cân bằng nhà/khách.** Mỗi đội đá sân nhà đúng $W/2$ trận.

**Break là gì.** Với một đội, nhìn dãy tuần theo thứ tự và ghi H (đá nhà) hay A
(đá khách): ta được một xâu độ dài $W$, ví dụ `H H A A A H ...`. **Một break là
một cặp tuần liên tiếp có cùng ký tự** — đội hai tuần liền đều đá nhà, hoặc hai
tuần liền đều đá khách. Xâu trên có 3 break (vị trí 1–2, 3–4, 4–5).

> Bản mô tả của IBM gọi "break" là *cả một đoạn* tuần liên tiếp cùng loại, nhưng
> công thức trong code — `teamBreaks[t] == sum(w in 2..nbWeeks)(playHome[t][w-1] == playHome[t][w])`
> — đếm theo **cặp liền kề**. Bài này dùng định nghĩa theo code.

5. **Cấm break dài.** Không đội nào được có ba tuần liên tiếp cùng loại (ba trận
   nhà liền, hoặc ba trận khách liền).
6. **Số break của mỗi đội phải chẵn.** Đây là hệ quả hình thức của (3) — xem ghi
   chú ở (C12) — bản gốc khai báo tường minh vì nó giúp engine cắt nhánh sớm.

**Mục tiêu.** Cực tiểu **tổng số break của toàn giải**, $\min\sum_t\beta_t$. Lịch
càng ít break thì càng "đảo" đều nhà–khách, càng công bằng cho khán giả và cho
việc di chuyển của các đội.

---

## (b) Mô hình toán học

Mô hình dưới đây là **hợp đồng chung** mà cả ba chiều cùng cài đặt. Ba đoạn code ở mục (c) chỉ là ba cách diễn đạt đúng mô hình này.

In [3]:
show_notes("3.1_sports", "Mô hình toán")

## (b) Mô hình toán học — dùng chung cho chiều `opl` và chiều `ortools`

### Tập hợp

| Ký hiệu | Định nghĩa | Ý nghĩa |
|---|---|---|
| $\mathcal{T}$ | $\{1,\dots,n\}$ | tập đội, $n$ chẵn |
| $\mathcal{W}$ | $\{1,\dots,W\}$, $W=2(n-1)$ | tập tuần |
| $\mathcal{G}$ | $\{1,\dots,\gamma\}$, $\gamma=n/2$ | tập chỗ đá trong một tuần |
| $\mathcal{M}$ | $\{1,\dots,M\}$, $M=n(n-1)$ | tập **trận** — mỗi cặp có thứ tự (nhà, khách) một trận |
| $\mathcal{S}$ | $\{1,\dots,M\}$ | tập **chỗ đá** của cả mùa, đánh số phẳng qua các tuần |
| $\mathcal{P}$ | $\{\{i,j\}: i,j\in\mathcal{T},\ i<j\}$ | tập cặp đối thủ, $|\mathcal{P}|=\binom n2$ |

Hai tập $\mathcal{M}$ và $\mathcal{S}$ **cùng lực lượng** $M$ — đó là điều kiện để
ràng buộc song ánh (C3) dùng được.

### Tham số

$$W = 2(n-1),\qquad \gamma=\frac n2,\qquad M=n(n-1)$$

$$\mu \;=\; \frac W2 + 1 \quad(\text{tuần đầu tiên của nửa sau mùa giải}),
\qquad
\delta \;=\;\begin{cases}\min\bigl(\tfrac n2,\,6\bigr) & n\ge 6\\[2pt] 0 & n<6\end{cases}$$

**Phép đánh số chỗ đá.** Chỗ đá thứ $g$ của tuần $w$ có số hiệu phẳng

$$s(w,g) \;=\; (w-1)\,\gamma + g \;\in\;\mathcal{S}$$

**Phép đánh số trận.** Cặp có thứ tự $(h,a)$ với $h\ne a$ ứng với đúng một trận

$$\iota(h,a) \;=\; (h-1)(n-1) + a - \mathbb{1}[a>h] \;\in\;\mathcal{M}$$

$\iota$ là **song ánh** từ $\{(h,a): h\ne a\}$ sang $\mathcal{M}$. Tập bộ ba hợp lệ:

$$\Pi \;=\;\bigl\{\,(h,\,a,\,\iota(h,a)) \;:\; h,a\in\mathcal{T},\ h\ne a\,\bigr\},
\qquad |\Pi| = M$$

### Biến quyết định

| Biến | Miền | Ý nghĩa |
|---|---|---|
| $x_{w,g}$ | $\mathcal{M}$ | số hiệu **trận** được xếp vào chỗ đá $g$ của tuần $w$ |
| $h_{w,g}$ | $\mathcal{T}$ | đội **nhà** ở chỗ đá đó |
| $a_{w,g}$ | $\mathcal{T}$ | đội **khách** ở chỗ đá đó |
| $\sigma_m$ | $\mathcal{S}$ | **chỗ đá** (phẳng) của trận $m$ — biểu diễn kép của $x$ |
| $\omega_m$ | $\mathcal{W}$ | **tuần** diễn ra trận $m$ |
| $p_{t,w}$ | $\{0,1\}$ | $=1$ khi và chỉ khi đội $t$ đá **sân nhà** ở tuần $w$ |
| $\beta_t$ | $\{0,1,\dots,W/2\}$ | số break của đội $t$ |

Bốn nhóm biến đầu là **bốn góc nhìn dư thừa** vào cùng một lời giải: nhìn theo chỗ
đá ($x,h,a$), nhìn theo trận ($\sigma$), nhìn theo tuần ($\omega$), nhìn theo đội
($p$). Dư thừa có chủ ý — mỗi góc nhìn cho engine một kênh lan truyền riêng, và
(C1)(C3)(C4)(C7) là các **ràng buộc kênh** (channelling) buộc bốn góc nhìn khớp nhau.
Đây là mẫu thiết kế điển hình của CP; mô hình MILP tương đương sẽ phải chọn đúng
một cách mã hoá.

### Ràng buộc

**Kênh giữa (nhà, khách) và số hiệu trận** — ràng buộc bảng:

$$\bigl(h_{w,g},\;a_{w,g},\;x_{w,g}\bigr)\;\in\;\Pi
\qquad\forall w\in\mathcal{W},\ g\in\mathcal{G} \tag{C1}$$

(C1) một mình gánh ba việc: cấm $h_{w,g}=a_{w,g}$ (đội không tự đá với mình), **định
nghĩa** $x_{w,g}=\iota(h_{w,g},a_{w,g})$ mà không cần viết công thức số học, và giữ
cho $x$ luôn nằm trong miền hợp lệ.

**Mỗi đội đá đúng một trận mỗi tuần:**

$$\operatorname{alldiff}\bigl(h_{w,1},\dots,h_{w,\gamma},\;a_{w,1},\dots,a_{w,\gamma}\bigr)
\qquad\forall w\in\mathcal{W} \tag{C2}$$

$2\gamma = n$ giá trị đôi một khác nhau lấy trong tập $n$ phần tử $\mathcal{T}$ ⇒
mỗi đội xuất hiện **đúng một lần** mỗi tuần. Đây là chỗ `allDifferent` mạnh hơn
$\binom{n}{2}$ bất đẳng thức $\ne$ rời rạc: bộ lọc miền của nó suy ra được điều đó
ngay ở mức lan truyền.

**Song ánh chỗ đá ↔ trận:**

$$x_{w,g}=m \;\Longleftrightarrow\; \sigma_m = s(w,g)
\qquad\forall w,g,\ \forall m\in\mathcal{M} \tag{C3}$$

(C3) chính là ràng buộc `inverse`. Nó là chỗ **thể thức double round-robin được
phát biểu**: vì $|\mathcal{M}|=|\mathcal{S}|=M$ và quan hệ là song ánh, mỗi trận
$m\in\mathcal{M}$ xuất hiện **đúng một lần** trong cả mùa. Cùng với (C1) và song
ánh $\iota$, điều đó nói: mỗi cặp có thứ tự (nhà, khách) đá đúng một trận, tức mỗi
cặp đội gặp nhau đúng hai lần, mỗi sân một lần. **Không cần viết thêm ràng buộc nào
cho thể thức giải đấu.**

**Tuần của một trận:**

$$\omega_m \;=\; \Bigl\lfloor \frac{\sigma_m-1}{\gamma}\Bigr\rfloor + 1
\qquad\forall m\in\mathcal{M} \tag{C4}$$

**Hai lượt ở hai nửa mùa giải.** Với mọi cặp $\{i,j\}\in\mathcal{P}$, đặt
$m_1=\iota(i,j)$ (i đá nhà) và $m_2=\iota(j,i)$ (j đá nhà):

$$\bigl[\omega_{m_1}\ge\mu\bigr] \;=\; \bigl[\omega_{m_2}<\mu\bigr] \tag{C5}$$

$$\bigl|\,\omega_{m_1}-\omega_{m_2}\,\bigr| \;\ge\; \delta \tag{C6}$$

trong đó $[\,\cdot\,]$ là hàm chỉ báo (nhận giá trị 0/1). (C5) đọc là: *đúng một
trong hai lượt nằm ở nửa sau mùa giải.*

**Kênh sang góc nhìn đội — ràng buộc đếm:**

$$p_{t,w} \;=\; \bigl|\{\,g\in\mathcal{G} \;:\; h_{w,g}=t\,\}\bigr|
\qquad\forall t\in\mathcal{T},\ w\in\mathcal{W} \tag{C7}$$

Vế phải là số lần đội $t$ xuất hiện trong dãy đội nhà của tuần $w$. Vì $p_{t,w}$
khai báo là biến **nhị phân**, (C7) đồng thời ép số đó $\le 1$ — điều này đã tự
đúng nhờ (C2), nhưng viết như vậy cho engine biết ngay.

**Cấm ba tuần liên tiếp cùng loại:**

$$1 \;\le\; p_{t,w}+p_{t,w+1}+p_{t,w+2} \;\le\; 2
\qquad\forall t\in\mathcal{T},\ w\in\{1,\dots,W-2\} \tag{C8}$$

Tổng $=3$ nghĩa là ba trận nhà liền; tổng $=0$ nghĩa là ba trận khách liền.

**Đếm break:**

$$\beta_t \;=\; \sum_{w=2}^{W} \bigl[\,p_{t,w-1}=p_{t,w}\,\bigr]
\qquad\forall t\in\mathcal{T} \tag{C9}$$

**Mở màn sân nhà thì khép lại sân khách:**

$$p_{t,1}\;\ne\;p_{t,W} \qquad\forall t\in\mathcal{T} \tag{C10}$$

**Cân bằng nhà/khách:**

$$\sum_{w\in\mathcal{W}} p_{t,w} \;=\; \frac W2 \qquad\forall t\in\mathcal{T} \tag{C11}$$

**Số break của mỗi đội là số chẵn:**

$$\beta_t \equiv 0 \pmod 2 \qquad\forall t\in\mathcal{T} \tag{C12}$$

> (C11) và (C12) là **ràng buộc xúc tác** (bản gốc gọi là *catalyzing constraints*):
> chúng **không đổi tập nghiệm**, chỉ giúp engine cắt nhánh sớm.
> (C11) suy ra từ thể thức giải đấu: theo (C1)+(C3), đội $t$ tiếp đón mỗi đội
> trong $n-1$ đội còn lại đúng một lần, nên nó đá sân nhà đúng $n-1=W/2$ trận.
> Suy luận đó đi qua ràng buộc song ánh nên **không nằm trong tầm lan truyền cục
> bộ** của engine — viết thẳng ra thì rẻ hơn nhiều.
> (C12) suy ra từ (C10): xâu H/A độ dài $W$ có hai đầu khác nhau nên số vị trí
> **đổi ký tự** là số lẻ; số break $=(W-1)-\#\text{đổi}$ là số chẵn vì $W-1$ lẻ.

**Phá đối xứng.** Các đội hoán vị được cho nhau, và thứ tự các trận trong một tuần
là tuỳ ý. Hai họ ràng buộc sau cắt hai nhóm đối xứng đó:

$$h_{1,g}=2g-1,\qquad a_{1,g}=2g \qquad\forall g\in\mathcal{G} \tag{C13}$$

$$x_{w,g} \;>\; x_{w,g-1} \qquad\forall w\in\mathcal{W},\ g\in\{2,\dots,\gamma\} \tag{C14}$$

(C13) cố định hẳn tuần 1 (đội 1 tiếp đội 2, đội 3 tiếp đội 4, …) — vừa phá đối
xứng hoán vị đội, vừa phá đối xứng **lật gương** của cả lịch thi đấu. (C14) ép các
trận trong một tuần xếp tăng dần theo số hiệu.

### Hàm mục tiêu

$$\boxed{\;\min\;\; B \;=\; \sum_{t\in\mathcal{T}} \beta_t \;}$$

### Tóm tắt kích thước

| Đại lượng | Công thức | $n=6$ | $n=8$ | $n=10$ |
|---|---|---|---|---|
| Tuần $W$ | $2(n-1)$ | 10 | 14 | 18 |
| Trận $M$ | $n(n-1)$ | 30 | 56 | 90 |
| Biến (mô hình toán) | $3\gamma W + 2M + nW + n$ | 216 | 400 | 640 |
| $\log_2$ không gian tìm kiếm | — | **510.6** (đo được) | *≈1 090 (ước tính)* | *≈1 925 (ước tính)* |

Cột cuối là lý do bài này chạy ở $n=6$ chứ không phải $n=10$ như bản gốc — xem (d).

### Biến thể B — mô hình của mẫu DOcplex.cp chính thức

Mẫu notebook của IBM giải một bài **khác**, ngắn gọn như sau. Ký hiệu $K$ = số đội
mỗi bảng, $n=2K$, số lượt gặp nhau $R=2$, số tuần $W=(K-1)R+KR=4K-2$ (**trùng khít**
công thức $2(n-1)$ của biến thể A).

| Biến | Miền | Ý nghĩa |
|---|---|---|
| $y_{t_1,t_2,r}$ | $\{1,\dots,W\}$ | tuần diễn ra lượt $r\in\{0,1\}$ giữa $t_1$ và $t_2$ |

$$y_{t_1,t_2,r} = y_{t_2,t_1,r} \qquad\forall t_1\ne t_2,\ r \tag{B1}$$

$$\operatorname{alldiff}\bigl(\{y_{t_1,t_2,r} : t_2\ne t_1,\ r\}\bigr)\qquad\forall t_1 \tag{B2}$$

$$\sum_{t_1\ne t_2,\;r} \mathbb{1}\bigl[(t_1,t_2)\text{ cùng bảng}\bigr]\cdot
\bigl[\,y_{t_1,t_2,r}\in\{1,\dots,\lfloor W/2\rfloor\}\,\bigr] \;\ge\; \lfloor W/3\rfloor \tag{B3}$$

$$\max\;\sum_{t_1\ne t_2,\;r}\ \mathbb{1}\bigl[(t_1,t_2)\text{ khác bảng}\bigr]\cdot y_{t_1,t_2,r}$$

**Không có khái niệm sân nhà/sân khách, do đó không có break và không có hàm mục
tiêu break.** Đó là lý do objective của chiều `docplexcp` (208) không so được với
objective của hai chiều còn lại (12).

---

## (c) Cài đặt ba chiều

### OPL → engine CP Optimizer

In [4]:
show_dimension_code("3.1_sports", "opl")

**OPL → engine CP Optimizer · ✅ mẫu chính thức**

```opl
/* Bài 3.1 — Lập lịch thi đấu thể thao (double round-robin) | Chiều OPL (engine CP Optimizer)
 *
 * Nguồn: LẤY MẪU CHÍNH THỨC.
 *   IBM ILOG CPLEX Optimization Studio 22.2 — opl/examples/opl/sports/sports.mod
 *   Licensed Materials - Property of IBM. 5725-A06 5725-A29 5724-Y48 5724-Y49
 *   5724-Y54 5724-Y55. Copyright IBM Corporation 1998, 2026. All Rights Reserved.
 *   Bản gốc để đối chiếu giữ nguyên tại models/3.1_sports/opl/sports_reference.mod
 *   (`diff` xác nhận chỉ khác đúng hai chỗ ghi dưới đây).
 *
 * TOÀN BỘ phần dựng mô hình giữ NGUYÊN VĂN bản của IBM. Chỉ có hai thay đổi:
 *   1. `int n = 10;`  ->  `int n = ...;`   — đưa n ra file dữ liệu để cả ba chiều
 *      dùng chung đúng một nguồn (data/sports/sports.dat). KHÔNG đổi mô hình.
 *   2. Thêm khối `execute EMIT_RESULT` ở cuối — in dòng RESULT theo giao kèo của
 *      tools/runner.py. KHÔNG đụng vào ràng buộc nào.
 *
 * Chạy:  tools/oplrun.sh models/3.1_sports/opl/sports.mod data/sports/sports.dat
 *
 * BỐN RÀNG BUỘC TOÀN CỤC được dùng ở đây (chất liệu chính cho mục so sánh ENGINE
 * trong NOTES.md). Kiểm chứng thật: CP-SAT CÓ tương đương trực tiếp cho ba cái
 * đầu và THIẾU `count` — chi tiết ở bảng đối chiếu NOTES.md mục (d.1):
 *   allowedAssignments(tupleset, x, y, z)  ràng buộc bảng 3 ngôi: bộ ba
 *                                          (home, away, gameId) phải nằm trong
 *                                          tập <h,a,gameId> hợp lệ
 *   allDifferent(...)                      mỗi tuần 2*nbGamesPerWeek chỗ đá phải
 *                                          là 2*nbGamesPerWeek đội khác nhau
 *   inverse(f, g)                          song ánh slot <-> trận: f[s]=g <=> g[g]=s
 *   count(array, v)                        đếm số lần giá trị v xuất hiện, ở đây
 *                                          reify thành biến bool playHome[t][w]
 */

using CP;

/* ------------------------------------------------------------

Problem Description
-------------------

The problem involves finding a schedule for a sports league. The league has 10
teams that play games over a season of 18 weeks. Each team has a home arena and
plays each other team twice during the season, once in its home arena and once in
the opposing team's home arena. For each of these games, the team playing at its
home arena is referred to as the home team; the team playing at the opponent's
arena is called the away team. There are 90 games altogether.

Each of the 18 weeks in the season has five identical slots to which games can be
assigned. Each team plays once a week. For each pair of teams, these two teams are
opponents twice in a season; these two games must be scheduled in different halves
of the season. Moreover, these two games must be scheduled at least six weeks
apart. A team must play at home either the first or last week but not both.

A break is a sequence of consecutive weeks in which a team plays its games either
all at home or all away. No team can have a break of three or more weeks in it. The
objective in this problem is to minimize the total number of breaks the teams play.

------------------------------------------------------------ */

int n = ...;   // <-- THAY ĐỔI DUY NHẤT Ở PHẦN MÔ HÌNH: bản gốc ghi `int n = 10;`


assert(n%2 == 0);

int nbWeeks = 2 * (n - 1);
int nbGamesPerWeek = n div 2;
int nbGames = n * (n - 1);
float mid = nbWeeks / 2 + 1;
int overlap = (n>=6) ? minl(n div 2, 6) : 0;


dvar int games[1..nbWeeks][1..nbGamesPerWeek] in 1..nbGames;
dvar int home[1..nbWeeks][1..nbGamesPerWeek] in 1..n;
dvar int away[1..nbWeeks][1..nbGamesPerWeek] in 1..n;
dvar int weekOfGame[1..nbGames] in  1..nbWeeks;
dvar int allSlots[1..nbGames] in 1..nbGames;
dvar boolean playHome[1..n][1..nbWeeks];
dvar int allGames[1..nbGames] = all[1..nbGames](w in 1..nbWeeks, g in 1..nbGamesPerWeek) games[w][g];
dvar int teamBreaks[1..n] in 0..nbWeeks div 2;

//
// For each play slot, set up correspondence between game id,
// home team, and away team
tuple PlaySlotTuple {
   int home;
   int away;
   key int gameId;
};

{PlaySlotTuple} playSlots = {<h, a, (h-1) * (n-1) + a - (a > h)> | h, a in 1..n : a != h};


execute {
    cp.param.timeLimit=60;
    cp.param.logPeriod=10000;
    cp.param.DefaultInferenceLevel="Extended";
}


//
// Objective: minimize the number of `breaks'.  A break is
//            two consecutive home or away matches for a
//            particular team
dexpr int breakCount = sum(t in 1..n) teamBreaks[t];


minimize breakCount;
// minimize sum(t in 1..n) teamBreaks[t];
subject to {

   forall(w in 1..nbWeeks, g in 1..nbGamesPerWeek)
     allowedAssignments(playSlots, home[w][g], away[w][g], games[w][g]);


   //
   // All teams play each week
   //
   forall(w in 1..nbWeeks) {

     allDifferent(append(all(g in 1..nbGamesPerWeek) home[w][g],
                         all(g in 1..nbGamesPerWeek) away[w][g]));

   }


    //
    // Dual representation: for each game id, the play slot is maintained
    //
    inverse(all [1..nbGames](w in 1..nbWeeks, g in 1..nbGamesPerWeek) games[w][g], allSlots);
    forall(g in 1..nbGames)
      weekOfGame[g] == ((allSlots[g]-1) div nbGamesPerWeek) + 1;


    //
    // Two half schedules.  Cannot play the same pair twice in the same half.
    // Plus, impose a minimum number of weeks between two games involving
    // the same teams (up to six weeks)
    //
    forall (<i,j,g1> in playSlots, <j,i,g2> in playSlots  : i < j) {
       (weekOfGame[g1] >= mid) == (weekOfGame[g2] < mid);
       if (overlap != 0)
          abs(weekOfGame[g1] - weekOfGame[g2]) >= overlap;
    }



    //
    // Can't have three homes or three aways in a row.
    //
    forall (t in 1..n, w in 1..nbWeeks) {
       playHome[t][w] == count(all(g in 1..nbGamesPerWeek) home[w][g], t);
    }

    forall (t in 1..n, w in 1..nbWeeks - 2) {
       1 <= sum(k in w..w+2) playHome[t][k] <= 2;
    }

    //
    // If we start the season home, we finish away and vice versa.
    //
    forall(t in 1..n)
       teamBreaks[t] == sum(w in 2..nbWeeks) (playHome[t][w-1] == playHome[t][w]);

    forall (t in 1..n)
      playHome[t][1] != playHome[t][nbWeeks];


    //
    // Catalyzing constraints
    //

    // Each team plays home the same number of times as away
    forall (t in 1..n) {
       sum(w in 1..nbWeeks) playHome[t][w] == nbWeeks div 2;
    }

    // Breaks must be even for each team
    forall(t in 1..n)
       teamBreaks[t] % 2 == 0;



    //
    // Symmetry breaking constraints
    //
    // Teams are interchangeable.  Fix first week.
    // Also breaks reflection symmetry of the whole schedule.
    forall (g in 1..nbGamesPerWeek) {
       home[1][g] == g*2 - 1;
       away[1][g] == g*2;
    }


    // Order of games in each week is arbitrary.
    // Break symmetry by forcing an order.
    forall (w in 1..nbWeeks)
      forall(g in 2..nbGamesPerWeek)
        games[w][g] > games[w][g-1];


}

int oponent[1..n][1..nbWeeks];

int breaks = sum(t in 1..n) teamBreaks[t];



execute {

      writeln("Solution at " + breaks);
      for (var j= 1; j <= nbWeeks; j++) {
         write("Week " + j + ": ");
         if (j < 10) write (" ");
         for (var i = 1; i <= nbGamesPerWeek; i++) {
            if (home[j][i] >= 10)
              write(home[j][i]);
            else
              write(" " + home[j][i]);
            write("-");
            if (away[j][i] >= 10)
              write(away[j][i]);
            else
              write(away[j][i] + " ");
            write(" ");
         }
         writeln();
      }
      writeln("Team schedules");
      for (i = 1; i <= n; i++) {
         write("T " + i + ":  ");
         var prev = -1;
         var brks = 0;
         for (j = 1; j <= nbWeeks; j++) {
            for (var k = 1; k <= nbGamesPerWeek; k++) {
               if (home[j][k] == i) {
                  oponent[i][j] = away[j][k];
                  if (away[j][k] >= 10)
                    write(away[j][k] + "H ")
                  else
                    write(" " + away[j][k] + "H ");
                  brks += (prev == 0);
                  prev = 0;
               }
               if (away[j][k] == i) {
                  oponent[i][j] = home[j][k];
                  if (home[j][k] >= 10)
                    write(home[j][k] + "A ");
                  else
                    write(" " + home[j][k] + "A ");
                  brks += (prev == 1);
                  prev = 1;
               }
            }
         }
         writeln("  " + brks + " breaks");
      }
      writeln();
}




tuple solution2DimT{
	int i;
	int j;
	int value;
};
tuple solution1DimT{
	int i;
	int value;
};

{solution2DimT} gamesReport = {<i,j, games[i][j]> | i in 1..nbWeeks, j in 1..nbGamesPerWeek};
{solution2DimT} homeReport = {<i,j, home[i][j]> | i in 1..nbWeeks, j in 1..nbGamesPerWeek};
{solution1DimT} awayReport = {<i, away[i][j]> | i in 1..nbWeeks, j in 1..nbGamesPerWeek};
{solution1DimT} weekOfGameReport = {<i, weekOfGame[i]> | i in 1..nbGames};
{solution1DimT} allSlotsReport = {<i, allSlots[i]> | i in 1..nbGames};
{solution2DimT} playHomeReport = {<i,j, playHome[i][j]> | i in 1..n, j in 1..nbWeeks};
{solution1DimT} allGamesReport = {<i, allGames[i]> | i in 1..nbGames};
{solution1DimT} teamBreaksReport = {<i, teamBreaks[i]> | i in 1..n};


// ===== BỔ SUNG — dòng RESULT theo giao kèo của tools/runner.py =====
// Không đụng tới mô hình; chỉ đọc lại lời giải và số liệu của engine.
execute EMIT_RESULT {
  writeln("RESULT {\"status\":\"Optimal\""
        + ",\"objective\":" + breaks
        + ",\"solve_time_s\":" + cp.info.solveTime
        + ",\"branches\":" + cp.info.numberOfBranches
        + ",\"fails\":" + cp.info.numberOfFails
        + ",\"n\":" + n
        + ",\"nb_weeks\":" + nbWeeks
        + ",\"nb_games\":" + nbGames + "}");
}
```

### DOcplex.cp → engine CP Optimizer

In [5]:
show_dimension_code("3.1_sports", "docplexcp")

**Python (docplex.cp) → engine CP Optimizer · ✅ mẫu chính thức**

```python
"""Bài 3.1 — Lập lịch thi đấu thể thao | Chiều DOcplex.cp (engine CP Optimizer)

Nguồn: LẤY MẪU CHÍNH THỨC.
  IBMDecisionOptimization/docplex — examples/cp/jupyter/sports_scheduling.ipynb
  "Use decision optimization to help a sports league schedule its games"
  Copyright (c) 2017, 2018 IBM. IPLA licensed Sample Materials.
  Bản sao cục bộ: vendor/docplex/examples/cp/jupyter/sports_scheduling.ipynb

Phần DỰNG MÔ HÌNH giữ NGUYÊN VĂN các ô code của notebook (ô 7, 8, 16, 18, 20, 22,
24, 26). Chỉ bổ sung: cấu hình engine cục bộ, tham số n qua dòng lệnh, in lời giải
bằng text thay cho pandas/HTML của notebook, và dòng RESULT theo giao kèo của
tools/runner.py. KHÔNG thêm/bớt/sửa một ràng buộc nào.

Chạy:  python3 models/3.1_sports/docplexcp/sports_scheduling_cp.py [n]

╔══════════════════════════════════════════════════════════════════════════════╗
║ CẢNH BÁO QUAN TRỌNG — HAI MẪU CHÍNH THỨC CỦA IBM LÀ HAI BÀI TOÁN KHÁC NHAU   ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ sports.mod (chiều OPL)        : double round-robin có sân nhà/sân khách,      ║
║                                 mục tiêu MIN tổng số break.      -> biến thể A║
║ sports_scheduling.ipynb (file này): lịch NFL hai bảng đấu, không phân biệt    ║
║                                 sân nhà/khách theo tuần, mục tiêu MAX tổng    ║
║                                 tuần của các trận liên bảng.     -> biến thể B║
║                                                                              ║
║ IBM đặt cùng tên "sports scheduling" cho hai mô hình khác hẳn nhau. Vì vậy    ║
║ KHÔNG so objective của chiều này với hai chiều còn lại. Chi tiết ở NOTES.md   ║
║ mục (c). Bản port biến thể A sang docplex.cp — để so được TRỤC NGÔN NGỮ và    ║
║ TRỤC ENGINE trên cùng một bài — nằm ở sports_portA_cp.py cạnh file này.       ║
╚══════════════════════════════════════════════════════════════════════════════╝

GHI CHÚ NGÔN NGỮ (điểm so sánh với OPL):
  `allowed_assignments` ở đây được dùng như một BIỂU THỨC BOOL rồi nhân với hệ số
  và cộng vào một tổng:

      sum(intra(t1,t2) * allowed_assignments(plays[...], FIRST_HALF_WEEKS)) >= k

  Cùng cái tên đó trong sports.mod lại là RÀNG BUỘC BẢNG ba ngôi trên một tupleset.
  Một cái tên, hai chữ ký, hai ngữ nghĩa — và bản OPL không reify được nó thành số
  hạng của một tổng. Xem NOTES.md mục (d).
"""

import json
import pathlib
import sys
from collections import namedtuple

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[3] / "tools"))
import cpo_env  # noqa: F401  — trỏ docplex.cp sang cpoptimizer cục bộ

from docplex.cp.model import CpoModel, all_diff, allowed_assignments, integer_var, maximize

# ---------------------------------------------------------------------------
# n lấy từ dòng lệnh (mặc định 6) để BA CHIỀU DÙNG CHUNG cùng một số đội.
# Notebook gốc tham số hoá bằng nbTeamsInDivision; ở đây n = 2*nbTeamsInDivision.
# Với numberOfMatchesToPlay = 2, công thức số tuần của notebook
#     (k-1)*2 + k*2 = 4k-2
# TRÙNG KHÍT với công thức của sports.mod:  2*(n-1) = 2*(2k-1) = 4k-2.
# ⇒ n = 6 cho 10 tuần ở cả hai biến thể. Số tuần khớp nhau, bài toán thì không.
# ---------------------------------------------------------------------------
N_TEAMS_ARG = int(sys.argv[1]) if len(sys.argv) > 1 else 6
assert N_TEAMS_ARG % 2 == 0, "n phải chẵn"
NB_TEAMS_IN_DIVISION = N_TEAMS_ARG // 2

# === PHẦN NGUYÊN VĂN NOTEBOOK — ô 7: dữ liệu đội =============================
# Teams in 1st division
TEAM_DIV1 = ["Baltimore Ravens", "Cincinnati Bengals", "Cleveland Browns", "Pittsburgh Steelers", "Houston Texans",
             "Indianapolis Colts", "Jacksonville Jaguars", "Tennessee Titans", "Buffalo Bills", "Miami Dolphins",
             "New England Patriots", "New York Jets", "Denver Broncos", "Kansas City Chiefs", "Oakland Raiders",
             "San Diego Chargers"]

# Teams in 2nd division
TEAM_DIV2 = ["Chicago Bears", "Detroit Lions", "Green Bay Packers", "Minnesota Vikings", "Atlanta Falcons",
             "Carolina Panthers", "New Orleans Saints", "Tampa Bay Buccaneers", "Dallas Cowboys", "New York Giants",
             "Philadelphia Eagles", "Washington Redskins", "Arizona Cardinals", "San Francisco 49ers",
             "Seattle Seahawks", "St. Louis Rams"]

# === PHẦN NGUYÊN VĂN NOTEBOOK — ô 8: tham số =================================
NUMBER_OF_MATCHES_TO_PLAY = 2  # Number of match to play between two teams on the league

T_SCHEDULE_PARAMS = (namedtuple("TScheduleParams",
                                ["nbTeamsInDivision",
                                 "maxTeamsInDivision",
                                 "numberOfMatchesToPlayInsideDivision",
                                 "numberOfMatchesToPlayOutsideDivision"
                                 ]))
# Schedule parameters: depending on their values, you may overreach the Community Edition of CPLEX
SCHEDULE_PARAMS = T_SCHEDULE_PARAMS(NB_TEAMS_IN_DIVISION,   # nbTeamsInDivision  (gốc: 5)
                                    10,  # maxTeamsInDivision
                                    NUMBER_OF_MATCHES_TO_PLAY,  # numberOfMatchesToPlayInsideDivision
                                    NUMBER_OF_MATCHES_TO_PLAY   # numberOfMatchesToPlayOutsideDivision
                                    )

# === PHẦN NGUYÊN VĂN NOTEBOOK — ô 16: chuẩn bị dữ liệu =======================
NB_TEAMS = 2 * SCHEDULE_PARAMS.nbTeamsInDivision
TEAMS = range(NB_TEAMS)

# Calculate the number of weeks necessary
NB_WEEKS = (SCHEDULE_PARAMS.nbTeamsInDivision - 1) * SCHEDULE_PARAMS.numberOfMatchesToPlayInsideDivision \
            + SCHEDULE_PARAMS.nbTeamsInDivision * SCHEDULE_PARAMS.numberOfMatchesToPlayOutsideDivision

# Weeks to schedule
WEEKS = tuple(range(NB_WEEKS))

# Season is split into two halves
FIRST_HALF_WEEKS = tuple(range(NB_WEEKS // 2))
NB_FIRST_HALS_WEEKS = NB_WEEKS // 3          # (lỗi chính tả "HALS" là của bản gốc)

# === PHẦN NGUYÊN VĂN NOTEBOOK — ô 18: biến quyết định ========================
mdl = CpoModel(name="SportsScheduling")

# Variables of the model
plays = {}
for i in range(NUMBER_OF_MATCHES_TO_PLAY):
    for t1 in TEAMS:
        for t2 in TEAMS:
            if t1 != t2:
                plays[(t1, t2, i)] = integer_var(1, NB_WEEKS, name="team1_{}_team2_{}_match_{}".format(t1, t2, i))

# === PHẦN NGUYÊN VĂN NOTEBOOK — ô 20: đối xứng cặp đấu =======================
# Constraints of the model
for t1 in TEAMS:
    for t2 in TEAMS:
        if t2 != t1:
            for i in range(NUMBER_OF_MATCHES_TO_PLAY):
                mdl.add(plays[(t1, t2, i)] == plays[(t2, t1, i)])  ### symmetrical match t1->t2 = t2->t1 at the ieme match

# === PHẦN NGUYÊN VĂN NOTEBOOK — ô 22: mỗi đội một trận mỗi tuần ==============
for t1 in TEAMS:
    mdl.add(all_diff([plays[(t1, t2, i)] for t2 in TEAMS if t2 != t1 for i in
                      range(NUMBER_OF_MATCHES_TO_PLAY)]))  ### team t1 must play one match per week

# === PHẦN NGUYÊN VĂN NOTEBOOK — ô 24: một phần trận nội bảng ở nửa đầu mùa ===
# Function that returns 1 if the two teams are in same division, 0 if not
def intra_divisional_pair(t1, t2):
    return int((t1 <= SCHEDULE_PARAMS.nbTeamsInDivision and t2 <= SCHEDULE_PARAMS.nbTeamsInDivision) or
               (t1 > SCHEDULE_PARAMS.nbTeamsInDivision and t2 > SCHEDULE_PARAMS.nbTeamsInDivision))

# Some intradivisional games should be in the first half
mdl.add(sum([intra_divisional_pair(t1, t2) * allowed_assignments(plays[(t1, t2, i)], FIRST_HALF_WEEKS)
             for t1 in TEAMS for t2 in [a for a in TEAMS if a != t1]
             for i in range(NUMBER_OF_MATCHES_TO_PLAY)]) >= NB_FIRST_HALS_WEEKS)

# === PHẦN NGUYÊN VĂN NOTEBOOK — ô 26: hàm mục tiêu ===========================
# Objective of the model is to schedule intradivisional games to be played late in the schedule
sm = []
for t1 in TEAMS:
    for t2 in TEAMS:
        if t1 != t2:
            if not intra_divisional_pair(t1, t2):
                for i in range(NUMBER_OF_MATCHES_TO_PLAY):
                    sm.append(plays[(t1, t2, i)])
mdl.add(maximize(sum(sm)))
# === HẾT PHẦN NGUYÊN VĂN =====================================================

msol = mdl.solve(TimeLimit=60, LogVerbosity="Quiet")

if msol:
    # Notebook gốc dựng bảng pandas + HTML; ở đây in text cho chạy được ngoài Jupyter.
    print(f"n = {NB_TEAMS} đội ({NB_TEAMS_IN_DIVISION} mỗi bảng), {NB_WEEKS} tuần")
    by_week = {w: [] for w in range(1, NB_WEEKS + 1)}
    for (t1, t2, i), var in plays.items():
        if t1 < t2:                     # mỗi cặp xuất hiện hai lần do ràng buộc đối xứng
            by_week[msol[var]].append((t1, t2, i, intra_divisional_pair(t1, t2)))
    for w in sorted(by_week):
        cells = " ".join(
            f"{t1}-{t2}{'*' if intra else ''}#{i}" for t1, t2, i, intra in sorted(by_week[w])
        )
        print(f"Week {w:>2}: {cells}")
    print("(* = trận nội bảng, #i = lượt đấu thứ i)")

infos = msol.get_solver_infos() if msol else {}
objs = msol.get_objective_values() if msol else None
print("RESULT " + json.dumps({
    "status": str(msol.get_solve_status()) if msol else "NoSolution",
    "objective": objs[0] if objs else None,
    "solve_time_s": round(msol.get_solve_time(), 4) if msol else None,
    "branches": infos.get("NumberOfBranches"),
    "fails": infos.get("NumberOfFails"),
    "n": NB_TEAMS,
    "nb_weeks": NB_WEEKS,
    "variant": "B (NFL hai bảng — max tổng tuần trận liên bảng); KHÔNG so objective với biến thể A",
}))
```

### OR-Tools → engine CP-SAT

In [6]:
show_dimension_code("3.1_sports", "ortools")

**Python (ortools.sat) → engine CP-SAT · ✍️ viết mới**

```python
"""Bài 3.1 — Lập lịch thi đấu thể thao (double round-robin) | Chiều OR-Tools (engine CP-SAT)

Nguồn: VIẾT MỚI. Google không phát hành ví dụ round-robin có break-minimization
trong bộ examples của OR-Tools; bản này là port ĐẦY ĐỦ mô hình `sports.mod` của
IBM (xem models/3.1_sports/opl/sports_reference.mod) sang CP-SAT.

Chạy:  python3 models/3.1_sports/ortools/sports_sat.py [data/sports/sports.dat]
       python3 models/3.1_sports/ortools/sports_sat.py --n 6 --workers 1

PHẠM VI: ĐẦY ĐỦ. Cả 14 ràng buộc của bản OPL đều được port, không lược bỏ ràng
buộc nào ⇒ objective (tổng số break) so trực tiếp được với chiều OPL.

╔══════════════════════════════════════════════════════════════════════════════╗
║ BỐN RÀNG BUỘC TOÀN CỤC CỦA sports.mod, DIỄN ĐẠT LẠI BẰNG CP-SAT              ║
╠════════════════════════╤═════════════════════════════════════════════════════╣
║ OPL / CP Optimizer     │ CP-SAT                                              ║
╠════════════════════════╪═════════════════════════════════════════════════════╣
║ allowedAssignments(    │ ✅ CÓ SẴN — add_allowed_assignments([h,a,g], tuples) ║
║   tupleSet, h, a, g)   │    Cùng ngữ nghĩa ràng buộc bảng. 0 biến phụ.       ║
╟────────────────────────┼─────────────────────────────────────────────────────╢
║ allDifferent(array)    │ ✅ CÓ SẴN — add_all_different(array). 0 biến phụ.    ║
╟────────────────────────┼─────────────────────────────────────────────────────╢
║ inverse(f, g)          │ ✅ CÓ SẴN — add_inverse(f, g).                       ║
║                        │ ⚠ CP-SAT bắt buộc miền 0..k-1; OPL đánh số 1..k     ║
║                        │    ⇒ phải đổi id trận sang gốc 0 khi dựng bảng.     ║
╟────────────────────────┼─────────────────────────────────────────────────────╢
║ count(array, t)        │ ❌ KHÔNG CÓ — phải diễn đạt vòng.                    ║
║  (reify thành          │    Mã hoá 1-hot toàn bộ home[w][g] bằng             ║
║   playHome[t][w])      │    add_map_domain, rồi playHome[t][w] = tổng bool.   ║
║                        │    Giá: n × nbWeeks × (n/2) biến bool phụ.          ║
╚════════════════════════╧═════════════════════════════════════════════════════╝

Ngoài bốn ràng buộc trên còn ba chỗ nữa CP-SAT phải diễn đạt vòng — xem các mục
(C8), (C11) và ghi chú `div`/`abs`/`%` trong thân file. Bảng đối chiếu đầy đủ kèm
số biến phụ đo được nằm ở NOTES.md mục (d).
"""

from __future__ import annotations

import argparse
import json
import pathlib
import sys
import time

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[3] / "tools"))
from opl_dat import load  # noqa: E402  — đọc đúng file .dat mà chiều OPL dùng

from ortools.sat.python import cp_model  # noqa: E402


def game_id(h: int, a: int, n: int) -> int:
    """Id trận (gốc 0) của cặp <đội nhà h, đội khách a>, đúng công thức của OPL.

    Bản gốc:  (h-1)*(n-1) + a - (a > h)      -> đánh số 1..n(n-1)
    Ở đây trừ thêm 1 vì `add_inverse` của CP-SAT BẮT BUỘC miền phải là 0..k-1,
    trong khi OPL cho phép miền 1..k. Đây là khác biệt kỹ thuật đầu tiên gặp phải
    khi port `inverse`.
    """
    return (h - 1) * (n - 1) + a - int(a > h) - 1


def build_and_solve(n: int, time_limit: float = 60.0, workers: int = 8, seed: int = 0,
                    verbose: bool = True) -> dict:
    assert n % 2 == 0, "n phải chẵn"

    # ---- tham số suy ra, đúng như sports.mod ---------------------------------
    nb_weeks = 2 * (n - 1)
    nb_games_per_week = n // 2
    nb_games = n * (n - 1)
    mid = nb_weeks // 2 + 1                       # tuần đầu tiên của nửa sau mùa giải
    overlap = min(n // 2, 6) if n >= 6 else 0     # khoảng cách tối thiểu giữa hai lượt

    WEEKS = range(nb_weeks)                       # 0-based nội bộ; in ra thì +1
    SLOTS = range(nb_games_per_week)
    TEAMS = range(1, n + 1)                       # 1-based, giữ như OPL

    model = cp_model.CpModel()
    nb_aux = 0                                    # đếm biến phụ do CP-SAT thiếu primitive

    # ---- biến quyết định (tương ứng 1-1 với sports.mod) ----------------------
    games = {(w, g): model.new_int_var(0, nb_games - 1, f"games_{w}_{g}")
             for w in WEEKS for g in SLOTS}
    home = {(w, g): model.new_int_var(1, n, f"home_{w}_{g}")
            for w in WEEKS for g in SLOTS}
    away = {(w, g): model.new_int_var(1, n, f"away_{w}_{g}")
            for w in WEEKS for g in SLOTS}
    # weekOfGame 0-based: tuần thật = week_of_game + 1
    week_of_game = [model.new_int_var(0, nb_weeks - 1, f"weekOfGame_{k}") for k in range(nb_games)]
    all_slots = [model.new_int_var(0, nb_games - 1, f"allSlots_{k}") for k in range(nb_games)]
    play_home = {(t, w): model.new_bool_var(f"playHome_{t}_{w}") for t in TEAMS for w in WEEKS}
    team_breaks = [model.new_int_var(0, nb_weeks // 2, f"teamBreaks_{t}") for t in TEAMS]

    # =========================================================================
    # (C1) allowedAssignments — RÀNG BUỘC BẢNG.  ✅ CP-SAT có sẵn.
    # Bộ ba (đội nhà, đội khách, id trận) phải nằm trong tập hợp lệ. Ràng buộc
    # này gánh luôn hai việc: cấm đội tự đá với mình, và ĐỊNH NGHĨA id trận theo
    # cặp (nhà, khách) — không cần viết công thức id thành ràng buộc số học.
    # =========================================================================
    play_slots = [(h, a, game_id(h, a, n)) for h in TEAMS for a in TEAMS if a != h]
    for w in WEEKS:
        for g in SLOTS:
            model.add_allowed_assignments([home[w, g], away[w, g], games[w, g]], play_slots)

    # =========================================================================
    # (C2) allDifferent — mỗi tuần, n chỗ đá (n/2 chủ nhà + n/2 khách) phải là n
    # đội khác nhau ⇒ mỗi đội đá đúng một trận mỗi tuần.  ✅ CP-SAT có sẵn.
    # =========================================================================
    for w in WEEKS:
        model.add_all_different([home[w, g] for g in SLOTS] + [away[w, g] for g in SLOTS])

    # =========================================================================
    # (C3) inverse — biểu diễn kép slot <-> trận.  ✅ CP-SAT có sẵn (add_inverse).
    # flat[s] = id trận xếp ở slot s;  all_slots[k] = slot của trận k.
    # ⚠ Khác biệt duy nhất: CP-SAT bắt hai mảng cùng kích thước và miền 0..k-1.
    # =========================================================================
    flat = [games[w, g] for w in WEEKS for g in SLOTS]   # slot s = w*nb_games_per_week + g
    model.add_inverse(flat, all_slots)

    # (C4) weekOfGame = allSlots div nbGamesPerWeek.
    # OPL viết thẳng `div` trên biểu thức biến; CP-SAT phải gọi ràng buộc chuyên
    # dụng add_division_equality (không có toán tử // trên IntVar).
    for k in range(nb_games):
        model.add_division_equality(week_of_game[k], all_slots[k], nb_games_per_week)

    # =========================================================================
    # (C5) Hai lượt của cùng một cặp đấu phải ở HAI NỬA MÙA khác nhau, và
    # (C6) cách nhau ít nhất `overlap` tuần.
    #
    # OPL viết:  (weekOfGame[g1] >= mid) == (weekOfGame[g2] < mid)
    # tức là reify hai bất đẳng thức rồi so bằng. CP-SAT không reify ngầm được:
    # phải tạo biến bool `second_half[k]` và buộc hai chiều bằng only_enforce_if.
    # Giá: nbGames biến bool phụ.
    # =========================================================================
    second_half = [model.new_bool_var(f"secondHalf_{k}") for k in range(nb_games)]
    nb_aux += nb_games
    for k in range(nb_games):
        # week_of_game là 0-based nên "nửa sau" là week_of_game >= mid-1
        model.add(week_of_game[k] >= mid - 1).only_enforce_if(second_half[k])
        model.add(week_of_game[k] <= mid - 2).only_enforce_if(~second_half[k])

    for i in TEAMS:
        for j in TEAMS:
            if i >= j:
                continue
            g1 = game_id(i, j, n)      # i đá sân nhà
            g2 = game_id(j, i, n)      # j đá sân nhà
            # đúng một trong hai lượt nằm ở nửa sau mùa giải
            model.add(second_half[g1] + second_half[g2] == 1)
            if overlap:
                # |w1 - w2| >= overlap. OPL có abs() ngay trong biểu thức ràng buộc;
                # CP-SAT phải vật chất hoá bằng add_abs_equality + 1 biến phụ.
                d = model.new_int_var(0, nb_weeks - 1, f"gap_{g1}_{g2}")
                nb_aux += 1
                model.add_abs_equality(d, week_of_game[g1] - week_of_game[g2])
                model.add(d >= overlap)

    # =========================================================================
    # (C7) playHome[t][w] == count(home[w][*], t)   ❌ CP-SAT KHÔNG CÓ `count`.
    #
    # Diễn đạt vòng: mã hoá 1-hot toàn bộ biến home[w][g] bằng add_map_domain
    #     is_home[w][g][t] = 1  <=>  home[w][g] == t
    # rồi playHome[t][w] = tổng các bool đó theo g. Vì mỗi tuần một đội đá nhiều
    # nhất một trận (C2), tổng này tự nằm trong {0,1} nên gán được cho biến bool.
    #
    # GIÁ PHẢI TRẢ: n × nbWeeks × (n/2) biến bool phụ + nbWeeks × (n/2) ràng buộc
    # exactly-one. Bản OPL: 0 biến phụ, `count` lo hết.
    # =========================================================================
    is_home = {}
    for w in WEEKS:
        for g in SLOTS:
            col = [model.new_bool_var(f"isHome_{w}_{g}_{t}") for t in TEAMS]
            nb_aux += n
            # add_map_domain: col[t-1] <=> (home[w][g] == t). offset=1 vì đội đánh số từ 1.
            model.add_map_domain(home[w, g], col, offset=1)
            for idx, t in enumerate(TEAMS):
                is_home[w, g, t] = col[idx]
    for t in TEAMS:
        for w in WEEKS:
            model.add(sum(is_home[w, g, t] for g in SLOTS) == play_home[t, w])

    # =========================================================================
    # (C8) Không được ba tuần liên tiếp cùng sân: 1 <= sum 3 tuần <= 2.
    # Thuần tuyến tính ⇒ cả hai engine viết như nhau.
    # =========================================================================
    for t in TEAMS:
        for w in range(nb_weeks - 2):
            model.add_linear_constraint(sum(play_home[t, k] for k in (w, w + 1, w + 2)), 1, 2)

    # =========================================================================
    # (C9) teamBreaks[t] = số tuần w mà playHome[t][w-1] == playHome[t][w].
    # OPL cộng thẳng biểu thức so sánh `(a == b)` vào một tổng — OPL tự reify.
    # CP-SAT không cho phép: mỗi số hạng phải là một biến bool được buộc hai chiều.
    # Giá: n × (nbWeeks-1) biến bool phụ.
    # =========================================================================
    brk = {}
    for t in TEAMS:
        for w in range(1, nb_weeks):
            b = model.new_bool_var(f"break_{t}_{w}")
            nb_aux += 1
            # b = 1  <=>  playHome[t][w-1] == playHome[t][w]
            model.add(play_home[t, w - 1] == play_home[t, w]).only_enforce_if(b)
            model.add(play_home[t, w - 1] + play_home[t, w] == 1).only_enforce_if(~b)
            brk[t, w] = b
        model.add(team_breaks[t - 1] == sum(brk[t, w] for w in range(1, nb_weeks)))

    # (C10) Mở màn sân nhà thì khép lại sân khách và ngược lại.
    for t in TEAMS:
        model.add(play_home[t, 0] + play_home[t, nb_weeks - 1] == 1)

    # ---- "Catalyzing constraints" của bản gốc: dư về logic, cần cho hiệu năng --
    # (C11) Số trận nhà = số trận khách.
    for t in TEAMS:
        model.add(sum(play_home[t, w] for w in WEEKS) == nb_weeks // 2)

    # (C12) Số break của mỗi đội phải chẵn.
    # OPL: `teamBreaks[t] % 2 == 0`. CP-SAT có add_modulo_equality nhưng cần một
    # biến đích ⇒ 1 biến phụ mỗi đội.
    for t in TEAMS:
        r = model.new_int_var(0, 1, f"breaksMod2_{t}")
        nb_aux += 1
        model.add_modulo_equality(r, team_breaks[t - 1], 2)
        model.add(r == 0)

    # ---- Phá đối xứng ---------------------------------------------------------
    # (C13) Đội hoán vị được cho nhau ⇒ cố định hẳn tuần 1.
    for g in SLOTS:
        model.add(home[0, g] == 2 * g + 1)
        model.add(away[0, g] == 2 * g + 2)

    # (C14) Thứ tự các trận trong một tuần là tuỳ ý ⇒ ép tăng dần theo id trận.
    for w in WEEKS:
        for g in range(1, nb_games_per_week):
            model.add(games[w, g] > games[w, g - 1])

    # ---- mục tiêu -------------------------------------------------------------
    total_breaks = sum(team_breaks)
    model.minimize(total_breaks)

    # ---- giải -----------------------------------------------------------------
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    # Cố định để số liệu tái lập được — xem PLAN.md §2.4.
    solver.parameters.num_workers = workers
    solver.parameters.random_seed = seed
    t0 = time.perf_counter()
    status = solver.solve(model)
    wall = time.perf_counter() - t0

    solved = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    obj = int(solver.value(total_breaks)) if solved else None

    if solved and verbose:
        print(f"Solution at {obj}")
        for w in WEEKS:
            cells = "  ".join(
                f"{solver.value(home[w, g]):>2}-{solver.value(away[w, g]):<2}" for g in SLOTS
            )
            print(f"Week {w + 1:>2}: {cells}")
        print("Team schedules")
        for t in TEAMS:
            row, brks, prev = [], 0, -1
            for w in WEEKS:
                for g in SLOTS:
                    if solver.value(home[w, g]) == t:
                        row.append(f"{solver.value(away[w, g]):>2}H")
                        brks += int(prev == 0)
                        prev = 0
                    elif solver.value(away[w, g]) == t:
                        row.append(f"{solver.value(home[w, g]):>2}A")
                        brks += int(prev == 1)
                        prev = 1
            print(f"T {t}:  " + " ".join(row) + f"   {brks} breaks")

    return {
        "status": solver.status_name(status),
        "objective": obj,
        "solve_time_s": round(solver.wall_time, 4),
        "branches": solver.num_branches,
        "fails": solver.num_conflicts,      # CP-SAT đếm conflicts, không phải fails
        "n": n,
        "nb_weeks": nb_weeks,
        "nb_games": nb_games,
        "aux_bool_vars": nb_aux,            # số biến phụ sinh ra vì CP-SAT thiếu primitive
        "wall_s": round(wall, 4),
        "variant": "A (sports.mod — min tổng break)",
    }


def _parse_args() -> argparse.Namespace:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("dat", nargs="*", default=None,
                    help="file .dat của OPL (mặc định data/sports/sports.dat)")
    ap.add_argument("--n", type=int, default=None, help="ghi đè số đội")
    ap.add_argument("--time-limit", type=float, default=60.0)
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--quiet", action="store_true")
    return ap.parse_args()


if __name__ == "__main__":
    a = _parse_args()
    if a.n is not None:
        n_teams = a.n
    else:
        files = a.dat or ["data/sports/sports.dat"]
        root = pathlib.Path(__file__).resolve().parents[3]
        n_teams = int(load(*[f if pathlib.Path(f).exists() else root / f for f in files])["n"])
    print("RESULT " + json.dumps(build_and_solve(
        n_teams, time_limit=a.time_limit, workers=a.workers, seed=a.seed, verbose=not a.quiet
    )))
```

## Chạy — cả ba chiều, cùng một bộ dữ liệu

Bảng dưới sinh ra bằng cách **chạy thật** ba file code vừa in ở trên.

In [7]:
run_table("3.1_sports")

,chiều,ngôn ngữ,engine,nguồn,biến thể,trạng thái,mục tiêu,thời gian giải (s),nhánh,fails/conflicts
0,opl,OPL,CP Optimizer,✅,A,Optimal,12,1.5510,556103,280303
1,docplexcp,Python (docplex.cp),CP Optimizer,✅,B,Feasible,208,60.0050,81371284,39598585
2,ortools,Python (ortools.sat),CP-SAT,✍️,A,OPTIMAL,12,0.7166,2216,0
3,docplexcp_portA,Python (docplex.cp),CP Optimizer,✍️,A,Optimal,12,2.2350,850352,429850


### Kiểm chứng chéo

Ba chiều phải cùng ra một nghiệm tối ưu. Không có bước này thì mọi so sánh hiệu năng đều vô nghĩa — nhanh hơn mà giải sai bài thì không nói lên điều gì.

In [8]:
cross_check("3.1_sports")

**Kiểm chứng chéo — bài 3.1_sports**

> ⚠️ Hai mẫu chính thức của IBM mang cùng tên 'sports scheduling' nhưng là HAI BÀI TOÁN KHÁC NHAU. KHÔNG so objective giữa chiều docplexcp và hai chiều còn lại.

**Biến thể A** — `opl` = 12 · `ortools` = 12 · `docplexcp_portA` = 12 → ✅ **KHỚP** — các chiều cùng một nghiệm tối ưu

**Biến thể B** — `docplexcp` = 208 → ⚠️ chỉ có một chiều, **không đối chiếu được**


## (d) Quan sát so sánh

In [9]:
show_notes("3.1_sports", "Quan sát")

## (d) Quan sát so sánh

### d.1 — Bảng đối chiếu ràng buộc toàn cục: CP Optimizer vs CP-SAT

Đây là điểm so sánh đắt nhất của bài. Cột "biến phụ" đếm ở $n=6$ (10 tuần, 30 trận).

| # | CP Optimizer (`sports.mod`) | CP-SAT tương ứng | Có sẵn? | Biến phụ ($n=6$) |
|---|---|---|---|---|
| (C1) | `allowedAssignments(playSlots, h, a, g)` — bảng 3 ngôi trên tupleset | `add_allowed_assignments([h,a,g], tuples)` | ✅ **1-1** | 0 |
| (C2) | `allDifferent(append(...))` | `add_all_different([...])` | ✅ **1-1** | 0 |
| (C3) | `inverse(allGames, allSlots)` | `add_inverse(flat, all_slots)` | ✅ **1-1**, nhưng CP-SAT **bắt buộc** miền $0..k-1$ còn OPL cho $1..k$ ⇒ phải dời gốc chỉ số | 0 |
| (C4) | `((allSlots[g]-1) div γ) + 1` — `div` viết thẳng trong biểu thức | `add_division_equality(ω, σ, γ)` — phải gọi ràng buộc chuyên dụng | ⚠️ có, khác cách viết | 0 |
| (C5) | `(ω[m1] >= μ) == (ω[m2] < μ)` — engine tự **reify** hai bất đẳng thức rồi so bằng | tạo bool $z_m$, buộc hai chiều bằng `only_enforce_if`, rồi `z[m1] + z[m2] == 1` | ❌ **phải diễn đạt vòng** | **30** bool |
| (C6) | `abs(ω[m1] - ω[m2]) >= δ` — `abs` là **biểu thức**, dùng ngay tại chỗ | `add_abs_equality(d, ω[m1]-ω[m2])` rồi `d >= δ` — `abs` phải **vật chất hoá** thành biến | ❌ phải thêm biến | **15** int |
| (C7) | `playHome[t][w] == count(home[w][*], t)` — `count` là **biểu thức số** | không có `count`. Mã hoá **1-hot** toàn bộ `home[w][g]` bằng `add_map_domain`, rồi $p_{t,w}=\sum_g$ bool | ❌ **phải diễn đạt vòng** | **180** bool |
| (C8) | `1 <= sum(...) <= 2` | `add_linear_constraint(sum, 1, 2)` | ✅ 1-1 | 0 |
| (C9) | `sum(w)(playHome[t][w-1] == playHome[t][w])` — cộng thẳng **biểu thức so sánh** vào một tổng | mỗi số hạng phải là bool được buộc hai chiều bằng `only_enforce_if` | ❌ phải diễn đạt vòng | **54** bool |
| (C10)(C11) | tuyến tính | tuyến tính | ✅ 1-1 | 0 |
| (C12) | `teamBreaks[t] % 2 == 0` — `%` là biểu thức | `add_modulo_equality(r, β, 2)` rồi `r == 0` — cần biến đích | ⚠️ có, cần biến phụ | **6** int |
| (C13)(C14) | gán/so sánh trực tiếp | như nhau | ✅ 1-1 | 0 |
| | | | | **Tổng: 285** |

**Kết luận rút ra từ bảng.**

**1. Ba trong bốn ràng buộc toàn cục "đặc thù CP Optimizer" hoá ra CP-SAT cũng có.**
Giả định ban đầu ở `PLAN.md` §4 — *"`sports.mod` dùng `allowedAssignments`, `inverse`,
`count` — CP-SAT không có tương đương trực tiếp"* — **chỉ đúng một phần ba**.
`add_allowed_assignments`, `add_all_different`, `add_inverse` đều tồn tại và cùng
ngữ nghĩa. Chỉ có **`count` là thật sự thiếu**.

**2. Nhưng khoảng cách thật không nằm ở danh mục ràng buộc — nó nằm ở khả năng
REIFY.** Nhìn cột "biến phụ": bốn dòng tốn kém nhất là (C5), (C6), (C7), (C9), và
cả bốn có chung một nguyên nhân. Trong OPL/docplex.cp, **một ràng buộc cũng là một
biểu thức**: `abs(...)`, `count(...)`, `(a == b)`, `(ω >= μ)` dùng được ngay tại
chỗ như một số hạng. Trong CP-SAT, mọi thứ như vậy phải **vật chất hoá thành biến
rồi buộc hai chiều bằng tay**. Đó mới là khác biệt engine, không phải chuyện thiếu
vài cái tên hàm.

**3. Cái giá đo được: +132% số biến.**

| | biến | ràng buộc | log₂ không gian tìm kiếm |
|---|---|---|---|
| CP Optimizer (`sports.mod`, $n=6$) | **216** | 271 | 510.6 (đo từ log engine) |
| CP Optimizer (`sports_portA_cp.py`) | **216** | 259 | 510.6 |
| CP-SAT (`sports_sat.py`) | **501** | 808 | — (CP-SAT không báo đại lượng này) |

$501 = 216 + 285$ — **khớp đúng đến từng biến** với cột "biến phụ" của bảng trên.
Ở $n=10$ con số biến phụ là **1 215**.

**4. `add_map_domain` là công cụ chuẩn để thay `count`.** Không có `count`, cách
diễn đạt vòng chuẩn là mã hoá 1-hot: `add_map_domain(home[w][g], col, offset=1)`
tạo $n$ bool với `col[t-1] ⇔ home[w][g] == t`, rồi $p_{t,w}=\sum_g \text{col}_t$.
Giá là $W\cdot\gamma\cdot n$ bool — $10\cdot3\cdot6=180$ ở $n=6$, tăng theo
$\Theta(n^3)$. Đây là dòng đắt nhất bảng, và cũng là ví dụ sạch nhất cho luận điểm
"CP-SAT quy mọi thứ về SAT nên phải bung ra biến bool".

### d.2 — Trục NGÔN NGỮ: OPL vs DOcplex.cp (cùng engine CP Optimizer, cùng bài, $n=6$)

Hai bản dựng **cùng 216 biến, cùng $\log_2$ không gian tìm kiếm 510.6**, chạy với
**cùng tham số engine** (`TimeLimit=60`, `DefaultInferenceLevel="Extended"` — đúng
những gì khối `execute` của `sports.mod` đặt). Trung vị 3 lần chạy:

| | Obj | Ràng buộc | Thời gian | Nhánh | Fails |
|---|---|---|---|---|---|
| `opl/sports.mod` | 12 | 271 | **1.51 s** | **556 103** | **280 303** |
| `docplexcp/sports_portA_cp.py` | 12 | 259 | 2.37 s | 850 352 | 429 850 |
| tỉ lệ | = | | **1.57× chậm hơn** | **1.53× nhiều nhánh hơn** | 1.53× |

Cả hai bản đều **tất định tuyệt đối** — ba lần chạy cho đúng cùng một con số nhánh.

**Đọc kết quả.** Cùng engine, cùng mô hình toán, cùng số biến, cùng không gian tìm
kiếm — mà lệch 1.5× cả thời gian lẫn số nhánh. Nguyên nhân là **thứ tự và cách bung
ràng buộc của hai front-end khác nhau** (271 so với 259 ràng buộc sau khi trích
xuất), đủ để đổi thứ tự duyệt của heuristic mặc định. Đây là **dẫn chứng thứ hai**
cho luận điểm "khác biệt do ngôn ngữ" của báo cáo, sau dẫn chứng N-Queens ở
`PLAN.md` §2.2 (440/198 nhánh của OPL so với 255/99 của DOcplex.cp).

Khác biệt so với bài N-Queens: ở đó OPL **buộc phải** viết dài hơn (`allDifferent`
không nhận mảng biểu thức). Ở bài này thì ngược lại — **OPL diễn đạt được mọi thứ
gọn hơn hoặc bằng** docplex.cp:

| Ý | OPL | docplex.cp |
|---|---|---|
| bảng 3 ngôi | `allowedAssignments(playSlots, h, a, g)` — nhận thẳng một `{PlaySlotTuple}` có `key` | `mdl.allowed_assignments([h,a,g], tuples)` — phải tự dựng `list[tuple]` |
| gốc chỉ số của `inverse` | theo **range khai báo** của mảng (ở đây $1..M$) | **cố định 0-based** — phải dời gốc id trận |
| bất đẳng thức kép | `1 <= expr <= 2` viết tự nhiên | phải gọi `mdl.range(expr, 1, 2)` |
| tổng có điều kiện | `sum(w in 2..W)(p[w-1] == p[w])` | `sum((p[w-1] == p[w]) for w in ...)` — tương đương |

⇒ Kết luận cho báo cáo: **ưu thế cú pháp không cố định về một phía.** OPL thắng ở
bài giàu tupleset và range (3.1); Python thắng ở bài cần mảng biểu thức (1.2).

### d.3 — Trục ENGINE: CP Optimizer vs CP-SAT (cùng ngôn ngữ Python, cùng bài, $n=6$)

| | Engine | Obj | Biến | Thời gian | Nhánh | Fails / Conflicts |
|---|---|---|---|---|---|---|
| `sports_portA_cp.py` | CP Optimizer | **12** | 216 | 2.37 s | 850 352 | 429 850 |
| `sports_sat.py` (8 worker) | CP-SAT | **12** | 501 | **0.78 s** | 1 350 | 0 |
| `sports_sat.py` (1 worker, seed 0) | CP-SAT | **12** | 501 | 2.59 s | 80 264 | 9 150 |

> `fails` của CP Optimizer và `conflicts` của CP-SAT đếm hai thứ khác nhau, chỉ so
> được trong cùng một engine (xem `README.md`).

**1. Hai lối tìm kiếm khác hẳn nhau.** CP-SAT một worker duyệt **80 264** nhánh,
CP Optimizer duyệt **850 352** — gấp **10.6 lần** — mà hai bên về đích gần như cùng
lúc (2.59 s so với 2.37 s). CP Optimizer duyệt ồ ạt với chi phí mỗi nhánh rất rẻ
(**359 000 nhánh/giây**); CP-SAT duyệt ít hơn hẳn nhờ học mệnh đề xung đột nhưng mỗi
nhánh đắt hơn (**31 000 nhánh/giây**). Đúng cùng một kết luận đã thấy ở bài 3.2.

**2. Ưu thế của CP-SAT ở đây là ĐA LUỒNG, không phải mỗi-nhánh-thông-minh-hơn.**
Bật 8 worker, CP-SAT về đích trong 0.78 s và chỉ duyệt ~1 350 nhánh — nhanh hơn 3.3×
so với chính nó chạy một luồng. Các worker chạy chiến lược khác nhau (LNS, no-LP,
core-based…) và worker nào gặp may thì kéo cả nhóm về đích. CP Optimizer bản
Community cũng dùng 16 worker song song (log ghi rõ) nhưng không thu được lợi tương
đương trên bài này.

**3. Giá phải trả: số liệu CP-SAT không tái lập được.** Ba lần chạy 8 worker cho
**826 / 1 350 / 2 648** nhánh và **0 / 0 / 41** conflicts, trong khi objective và
thời gian gần như không đổi. Cố định `num_workers=1` + `random_seed=0` thì số liệu
tất định tuyệt đối (80 264 / 9 150 ở cả ba lần). Đây là **xác nhận độc lập** cho
`PLAN.md` §2.4 trên một bài toán thứ hai ⇒ mọi bảng benchmark của báo cáo bắt buộc
phải cố định worker và seed.

### d.4 — Trần Community Edition: chỗ CP-SAT thắng tuyệt đối

Bản gốc của IBM đặt $n=10$. Trên **CPLEX Studio Community Edition** thì cả hai chiều
CP Optimizer đều **không chạy nổi bản gốc**:

| $n$ | Tuần | log₂ KGTK | OPL / DOcplex.cp (CP Optimizer Community) | OR-Tools (CP-SAT) |
|---|---|---|---|---|
| **6** | 10 | **510.6** (đo được) | ✅ tối ưu **12** — 1.51 s | ✅ tối ưu **12** — 0.78 s |
| 8 | 14 | ≈1 090 (ước tính) | ❌ `FATAL[ENGINE_001]: Problem size limit exceeded` | ✅ tối ưu **16** — 14.8 s |
| **10** *(cỡ bản gốc IBM)* | 18 | ≈1 925 (ước tính) | ❌ `FATAL[ENGINE_001]` | ✅ tối ưu **16** — 17.9 s |

Thông báo lỗi thật (cả `oplrun` lẫn `docplex.cp` đều báo giống nhau):

```
*** FATAL[ENGINE_001]: Exception from IBM ILOG Concert: Problem size limit exceeded.
CP Optimizer Community Edition solves problems with search spaces up to 2^1000.
```

$n=6$ là **giá trị lớn nhất chạy được** trên bản Community. Ba nhận xét:

1. **Trần 2^1000 là trần LICENSE, không phải trần thuật toán.** Với license
   academic thì $n=10$ chạy bình thường; đây là giới hạn thương mại của IBM.
2. **CP-SAT không có trần nào tương ứng** — và bản port có **nhiều hơn 132% số biến**
   mà vẫn giải được cỡ bài gấp ba. Nghịch lý biểu kiến này chính là điểm đáng nói:
   mã hoá "cồng kềnh" hơn không phải là bất lợi khi engine sinh ra để nuốt biến bool.
3. Đây là dẫn chứng cụ thể cho luận điểm **hạ tầng/giấy phép cũng là một chiều so
   sánh**, ngang hàng với ngôn ngữ và engine: `ortools` là thư viện Apache-2.0 cài
   bằng `pip install`, còn CP Optimizer đòi cài Studio và bị chặn theo cỡ bài.

### d.5 — Ghi chú về chiều `docplexcp` (biến thể B)

Chiều này giải bài khác nên không vào hai bảng trên. Số liệu để tham khảo ($n=6$,
$W=10$, 60 s):

| Status | Obj | Thời gian | Nhánh | Fails |
|---|---|---|---|---|
| Feasible (**không chứng minh được tối ưu**) | 208 | 60.0 s (chạm giới hạn) | **79 268 907** | 38 573 731 |

Chỉ 60 biến, mà CP Optimizer duyệt **79 triệu nhánh** trong 60 s vẫn không đóng được
bài. Lý do: mô hình (B) **không có ràng buộc phá đối xứng nào**, trong khi biến thể A
có tới hai họ ((C13)(C14)). Đặt cạnh nhau, đây là minh hoạ rất gọn cho vai trò của
phá đối xứng trong CP: bài 216 biến **có** phá đối xứng đóng trong 1.5 s; bài 60 biến
**không** phá đối xứng, 60 s vẫn treo. Nó cũng cho thấy **tốc độ duyệt nhánh khủng
khiếp của CP Optimizer** — 1.3 triệu nhánh/giây — và tốc độ đó một mình không cứu
được một mô hình dựng kém.

---

---

# Bài 3.2 — Xếp thời khoá biểu có ràng buộc khả dụng

## (a) Phát biểu bài toán

In [10]:
show_notes("3.2_timetable", "Phát biểu")

## (a) Phát biểu bài toán

Xếp thời khoá biểu cho một trường: mỗi lớp phải học đủ số tiết từng môn, mỗi môn
do một giáo viên có chuyên môn phù hợp dạy, học trong một phòng phù hợp. Giáo
viên, lớp và phòng đều chỉ ở một chỗ tại một thời điểm. Mục tiêu: **tối thiểu
makespan** — kết thúc toàn bộ chương trình sớm nhất có thể.

Phần bài 3.2 thêm vào so với bản gốc là **ràng buộc khả dụng** (availability
windows): giáo viên có lịch bận, lớp có lịch bận, và mỗi môn có khung giờ cần tránh.

## (b) Mô hình toán học

Mô hình dưới đây là **hợp đồng chung** mà cả ba chiều cùng cài đặt. Ba đoạn code ở mục (c) chỉ là ba cách diễn đạt đúng mô hình này.

In [11]:
show_notes("3.2_timetable", "Mô hình toán")

## (b) Mô hình toán học — dùng chung cho cả ba chiều

> Mô hình dưới đây mô tả đúng những gì ba chiều cài đặt. Nó **thay thế** bản phác
> thảo ở brief §4: bản đó dùng mã hoá nhị phân $X_{c,t,d,p}$ và bỏ sót phòng học,
> thời lượng, số lần lặp, giờ nghỉ và môn buổi sáng.

### Tập hợp

| Ký hiệu | Ý nghĩa |
|---|---|
| $\mathcal{C}$ | tập lớp |
| $\mathcal{S}$ | tập môn |
| $\mathcal{T}$ | tập giáo viên |
| $\mathcal{R}$ | tập phòng |
| $\mathcal{K}\subseteq\mathcal{T}\times\mathcal{S}$ | chuyên môn: $(t,s)\in\mathcal{K}$ nghĩa là $t$ dạy được $s$ |
| $\mathcal{D}\subseteq\mathcal{R}\times\mathcal{S}$ | phòng chuyên dụng: $(r,s)\in\mathcal{D}$ nghĩa là phòng $r$ dành cho môn $s$ |
| $\mathcal{B}\subseteq\mathcal{S}\times\mathcal{S}$ | cặp môn kỵ nhau, cần giãn cách |
| $\mathcal{M}\subseteq\mathcal{S}$ | môn chỉ được học buổi sáng |
| $\mathcal{Q}$ | chương trình học |

### Tham số

$$H=\text{số tiết mỗi ngày},\quad H_2=\tfrac{H}{2},\quad N_d=\text{số ngày},\quad
T_{\max}=H\!\cdot\!N_d,\quad \beta=\text{độ dài giãn cách}$$

Trục thời gian $\mathcal{H}=\{0,1,\dots,T_{\max}-1\}$, đánh số tiết liên tục qua các
ngày. Tiết $u$ thuộc ngày $\lfloor u/H\rfloor$ và buổi $\lfloor u/H_2\rfloor$.

### Từ chương trình học sang các buổi học

Mỗi phần tử $q=(c_q,\,s_q,\,p_q,\,n_q)\in\mathcal{Q}$ đọc là: *lớp $c_q$ phải học môn
$s_q$ thành $n_q$ buổi, mỗi buổi dài $p_q$ tiết.* Tách ra thành tập **buổi học**

$$\mathcal{I}=\bigl\{\,(q,k)\;:\;q\in\mathcal{Q},\ k\in\{1,\dots,n_q\}\,\bigr\}$$

Với $i=(q,k)\in\mathcal{I}$ viết gọn $c_i=c_q$, $s_i=s_q$, $p_i=p_q$, $q_i=q$, $k_i=k$.

### Tài nguyên dùng được

$$\mathcal{T}_s=\{t\in\mathcal{T}:(t,s)\in\mathcal{K}\}$$

$$\mathcal{R}_s=\underbrace{\{r:(r,s)\in\mathcal{D}\}}_{\text{phòng chuyên dụng của } s}
\;\cup\;
\underbrace{\{r: r\notin\Pi_\mathcal{R}(\mathcal{D})\ \wedge\ s\notin\Pi_\mathcal{S}(\mathcal{D})\}}_{\text{phòng thường, khi } s \text{ không đòi phòng riêng}}$$

trong đó $\Pi_\mathcal{R},\Pi_\mathcal{S}$ là phép chiếu $\mathcal{D}$ xuống thành phần
phòng và thành phần môn. Nói bằng lời: môn có phòng riêng thì chỉ học ở phòng riêng
đó; môn không đòi phòng riêng thì học ở bất kỳ phòng nào không bị dành riêng cho môn khác.

### Biến quyết định

| Biến | Miền | Ý nghĩa |
|---|---|---|
| $\sigma_i$ | $\mathcal{H}$ | tiết bắt đầu của buổi $i$ |
| $\eta_i$ | $\mathcal{H}$ | tiết kết thúc (hở phải): buổi $i$ chiếm $[\sigma_i,\eta_i)$ |
| $\theta_i$ | $\mathcal{T}_{s_i}$ | giáo viên dạy buổi $i$ |
| $\mu_i$ | $\mathcal{R}_{s_i}$ | phòng học buổi $i$ |
| $\gamma_{c,s}$ | $\mathcal{T}$ | giáo viên cố định của cặp (lớp $c$, môn $s$) |
| $\omega$ | $\mathcal{H}$ | makespan |

Việc $\theta_i$ và $\mu_i$ lấy miền $\mathcal{T}_{s_i}$, $\mathcal{R}_{s_i}$ đã **nuốt
luôn** hai ràng buộc "giáo viên phải đủ chuyên môn" và "phòng phải phù hợp" vào
trong khai báo biến — không cần viết thành ràng buộc riêng. Đây là nét đặc trưng
của CP đã gặp ở bài 1.2 với N-Queens.

> Lưu ý: $\eta_i\in\mathcal{H}$ nên $\eta_i\le T_{\max}-1$, tức tiết cuối cùng của kỳ
> không bao giờ được dùng. Đây là đặc điểm của bản OPL gốc; hai chiều còn lại giữ
> nguyên để ba chiều so được với nhau.

### Ràng buộc

**Cấu trúc thời gian**

$$\eta_i=\sigma_i+p_i \qquad \forall i\in\mathcal{I} \tag{C1}$$

$$\sigma_i<\sigma_j \qquad \forall i,j\in\mathcal{I}:\ q_i=q_j,\ k_i<k_j \tag{C2}$$

(C2) đánh số các buổi của cùng một môn theo đúng thứ tự thời gian — vừa hợp lẽ, vừa
phá đối xứng giữa các buổi giống hệt nhau, giúp engine chứng minh tối ưu nhanh hơn.

**Tài nguyên chỉ ở một chỗ tại một thời điểm.** Với mọi $i\ne j$:

$$c_i=c_j \;\Longrightarrow\; [\sigma_i,\eta_i)\cap[\sigma_j,\eta_j)=\varnothing \tag{C3}$$
$$\theta_i=\theta_j \;\Longrightarrow\; [\sigma_i,\eta_i)\cap[\sigma_j,\eta_j)=\varnothing \tag{C4}$$
$$\mu_i=\mu_j \;\Longrightarrow\; [\sigma_i,\eta_i)\cap[\sigma_j,\eta_j)=\varnothing \tag{C5}$$

Ba ràng buộc này là chỗ **ba chiều mã hoá khác nhau** — xem mục kế tiếp.

**Ổn định phân công**

$$\theta_i=\gamma_{c_i,\,s_i} \qquad \forall i\in\mathcal{I} \tag{C6}$$

Một lớp học một môn thì suốt kỳ chỉ một giáo viên dạy.

**Ràng buộc lịch**

$$p_i>1 \;\Longrightarrow\; \Bigl\lfloor \tfrac{\sigma_i}{H_2}\Bigr\rfloor=\Bigl\lfloor \tfrac{\eta_i-1}{H_2}\Bigr\rfloor \tag{C7}$$

$$s_i\in\mathcal{M} \;\Longrightarrow\; \sigma_i \bmod H < H_2 \tag{C8}$$

$$c_i=c_j,\ s_i=s_j,\ i\ne j \;\Longrightarrow\; \Bigl\lfloor\tfrac{\sigma_i}{H}\Bigr\rfloor\ne\Bigl\lfloor\tfrac{\sigma_j}{H}\Bigr\rfloor \tag{C9}$$

(C7) buổi dài hơn một tiết không được vắt qua giờ nghỉ trưa. (C8) môn buổi sáng phải
bắt đầu trong buổi sáng. (C9) một lớp không học cùng một môn hai lần trong một ngày.

**Giãn cách giữa hai môn kỵ nhau.** Với $i\ne j$, $c_i=c_j$, và $(s_i,s_j)\in\mathcal{B}$
hoặc $(s_j,s_i)\in\mathcal{B}$:

$$\Bigl\lfloor\tfrac{\sigma_i}{H}\Bigr\rfloor\ne\Bigl\lfloor\tfrac{\sigma_j}{H}\Bigr\rfloor
\;\;\vee\;\;
\Bigl\lfloor\tfrac{\sigma_i}{H_2}\Bigr\rfloor\ne\Bigl\lfloor\tfrac{\sigma_j}{H_2}\Bigr\rfloor
\;\;\vee\;\;
g_{ij}\ \ge\ \beta \tag{C10}$$

$$g_{ij}=\max(0,\ \sigma_i-\eta_j)+\max(0,\ \sigma_j-\eta_i)$$

Khác ngày, hoặc khác buổi, hoặc cách nhau đủ $\beta$ tiết. Do (C3) hai buổi cùng lớp
không chồng nhau nên nhiều nhất một số hạng của $g_{ij}$ dương — nó đúng bằng khoảng
trống giữa hai buổi.

### Phần bài 3.2 bổ sung — ràng buộc khả dụng

Dữ liệu vào thêm ba tập lịch bận:

$$\mathcal{U}^{T}\subseteq\mathcal{T}\times\mathcal{H},\qquad
\mathcal{U}^{C}\subseteq\mathcal{C}\times\mathcal{H},\qquad
\mathcal{U}^{S}\subseteq\mathcal{S}\times\mathcal{H}$$

lần lượt là lịch bận của giáo viên, lịch bận của lớp, và khung giờ cần tránh của môn.
Đặt vị từ "buổi $i$ **không** phủ tiết $u$":

$$\mathrm{free}(i,u)\;\equiv\;u\notin[\sigma_i,\eta_i)\;\equiv\;\bigl(\sigma_i>u\ \vee\ \eta_i\le u\bigr)$$

$$\theta_i=t \;\Longrightarrow\; \mathrm{free}(i,u) \qquad \forall i\in\mathcal{I},\ \forall (t,u)\in\mathcal{U}^{T} \tag{RB7}$$

$$\mathrm{free}(i,u) \qquad \forall i\in\mathcal{I},\ \forall (c,u)\in\mathcal{U}^{C}:\ c_i=c \tag{RB8}$$

$$\mathrm{free}(i,u) \qquad \forall i\in\mathcal{I},\ \forall (s,u)\in\mathcal{U}^{S}:\ s_i=s \tag{RB4}$$

Chú ý bất đối xứng: (RB7) phải viết dạng kéo theo vì $\theta_i$ là **biến**, còn
(RB8) và (RB4) lọc thẳng ở chỉ số vì $c_i,s_i$ là **hằng** của buổi học. Bất đối
xứng này hiện ra trong cả ba bản cài đặt.

### Hàm mục tiêu

$$\omega=\max_{i\in\mathcal{I}}\eta_i, \qquad
\omega\ \ge\ \max_{c\in\mathcal{C}}\sum_{i\,:\,c_i=c}p_i \tag{C11}$$

$$\boxed{\ \min\ \omega\ }$$

Bất đẳng thức trong (C11) là **chặn dưới hợp lệ**: một lớp có tổng $P$ tiết thì không
thể học xong trước tiết $P$, vì (C3) cấm lớp học hai buổi cùng lúc. Nó không đổi tập
nghiệm tối ưu, chỉ giúp engine chứng minh tối ưu sớm hơn.

### Ba chiều mã hoá (C3)–(C5) theo ba cách

Cùng một mô hình toán ở trên, ba chiều diễn đạt phần "tài nguyên dùng một lần tại
mỗi thời điểm" bằng ba cách khác nhau. **Đây là biến số mà bài 3.2 đo.**

**① OPL — đếm có điều kiện.** Hai đoạn giao nhau thì điểm bắt đầu muộn hơn luôn nằm
trong đoạn kia, nên (C3) tương đương: với mọi $i$, số buổi cùng lớp bắt đầu trong
$[\sigma_i,\eta_i)$ không quá 1.

$$\sum_{j\,:\,c_j=c_i}\mathbb{1}\bigl[\sigma_i\le\sigma_j<\eta_i\bigr]\ <\ 2 \qquad\forall i$$

Sinh $O(|\mathcal{I}|^2)$ tích có điều kiện — nguồn gốc của con số 17 s ở bảng benchmark.

**② DOcplex.cp — biến interval, `alternative` + `no_overlap`.** Mỗi buổi là một biến
interval $x_i$; mỗi tài nguyên dùng được sinh một interval tuỳ chọn; `alternative`
chọn đúng một.

$$\texttt{no\_overlap}\bigl(\{x_i : c_i=c\}\bigr)\quad\forall c\in\mathcal{C}$$

**③ OR-Tools — biến interval + literal hiện diện.** Cùng ý tưởng interval, nhưng
CP-SAT không có `alternative` nên phải dựng tay: $\pi_{i,t}\in\{0,1\}$ với
$\pi_{i,t}=1\iff\theta_i=t$, rồi `add_no_overlap` trên các interval tuỳ chọn.

Ba cách **tương đương về tập nghiệm** — đã kiểm chứng chéo ở mục dưới — nhưng khác
hẳn về sức lan truyền và chi phí mỗi nhánh.

## (c) Cài đặt ba chiều

### OPL → engine CP Optimizer

In [12]:
show_dimension_code("3.2_timetable", "opl")

**OPL → engine CP Optimizer · ✅+✍️ mẫu chính thức, có mở rộng**

```opl
// --------------------------------------------------------------------------
// Licensed Materials - Property of IBM
//
// 5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55
// Copyright IBM Corporation 1998, 2026. All Rights Reserved.
//
// Note to U.S. Government Users Restricted Rights:
// Use, duplication or disclosure restricted by GSA ADP Schedule
// Contract with IBM Corp.
// --------------------------------------------------------------------------

using CP;

/* 
 This model solves a school time tabling problem.
 Given teacher skills, room equipment and pupil course requirement,
 the model generates for each course a time table specifying :
    - a teacher
    - a start time
    - a room
  
constraints are used to:

   - ensure the course ends after it starts         
   - ensure course numerotation is chronological     
   - ensure that a teacher is required once at any time point.  
   - ensure the teacher can teach the discipline 
   - ensure that a room is required once at any time point.
   - ensure the room can support the discipline
   - ensure that a class follows one course at a time      
   - ensure that for given class and discipline, the teacher is always the same  
   - ensure a course starts and end the same halfday
   - insert break duration between specified disciplines
   - avoid to have the same discipline taught twice a day
   - ensure that the morning disciplines end in the morning


Note: To reduce the amount of decision variable, we choose to use
course start times as time points where uniqueness of resources (classes, 
teachers and rooms) is enforced.

This model is greater than the size allowed in trial mode. 
If you want to run this example with the large data set, you need a commercial edition of CPLEX Studio to run this example. 
If you are a student or teacher, you can also get a full version through
the IBM Academic Initiative. 
*/
  
execute{
	}
	
tuple Pair {
  string a;
  string b;
};
tuple Requirement {
   string Class;            // a set of pupils
   string discipline;       // what will be taught
   int    Duration;         // course duration
   int    repetition;       // how many time the course is repeated
};
//
// user given model data
//

{Pair} NeedBreak = ...;                   // disciplines that should not be contiguous in time
{string} MorningDiscipline = ...;         // disciplines that must be taught in the morning
{Pair} TeacherDisciplineSet = ...;        // what are the teacher skills
{Pair} DedicatedRoomSet = ...;            // a set of disciplines requiring special rooms
{Requirement} RequirementSet = ...;       // the educational program
{string} Room = ...;                      // the set of available rooms
int BreakDuration = ...;                  // time interval between two disciplines
int DayDuration = ...;                    // must be even (morning duration equals afternoon duration)
int NumberOfDaysPerPeriod = ...;          // how many worked days per period

// ===== BỔ SUNG cho bài 3.2 — ràng buộc khả dụng (availability windows) =====
// Ba tập dưới đây là phần bài 3.2 thêm vào so với timetabling.mod gốc của IBM.
// Mô hình gốc ngầm giả định mọi tài nguyên luôn rảnh trong toàn bộ kỳ.
tuple Unavailable {
  string who;   // tên giáo viên / lớp / môn, tuỳ tập chứa nó
  int    t;     // thời điểm bận, tính theo Time = 0..MaxTime-1
};
{Unavailable} TeacherBusySet     = ...;   // RB7 — giáo viên bận
{Unavailable} ClassBusySet       = ...;   // RB8 — lớp bận
{Unavailable} DisciplineAvoidSet = ...;   // RB4 — khung giờ cần tránh cho môn
// ===== hết phần bổ sung =====
//
// vocabularies
//

{string} Class = {c | <c,d,u,n> in RequirementSet };
{string} Teacher = { t | <t,d> in TeacherDisciplineSet };
{string} Discipline =  {d | <t,d> in TeacherDisciplineSet };


//
// time expressions
//
int HalfDayDuration = DayDuration div 2;
int MaxTime = DayDuration*NumberOfDaysPerPeriod;
range Time = 0..MaxTime-1;
//
// convenience expressions for room compatibility
//
int PossibleRoom[d in Discipline, x in Room] = 
  <x,d> in DedicatedRoomSet 
  || 0 == card({<z,k> | z in Room, k in Discipline
               : (<x,k> in DedicatedRoomSet) 
                 || (<z,d> in DedicatedRoomSet)});
int NbRoom = card(Room);
range RoomId = 0..NbRoom-1;
{int} PossibleRoomIds[d in Discipline] = 
  {i | i in RoomId, z in Room
   :  (PossibleRoom[d,z] == 1) && (i == ord(Room,z))};
//
// convenience expressions for teacher skills
//

// possible teacher disciplines
{string} PossibleTeacherDiscipline[x in Teacher] = {d | <x,d> in TeacherDisciplineSet };
int NbTeacher = card(Teacher);
range TeacherId = 0..NbTeacher-1;

// possible teacher ids
{int} PossibleTeacherIds[d in Discipline] =
{i | i in TeacherId, z in Teacher 
   : i == ord(Teacher, z) 
     && d in PossibleTeacherDiscipline[z] };

//
// convenience expressions for requirement instantiation
//

// for a given requirement, an instance is one course occurrence
tuple Instance {
  string Class;
  string discipline;
  int    Duration;
  int    repetition;
  int    id;
  int    requirementId;
};
{Instance} InstanceSet = { 
  <c,d,t,r,i,z> | <c,d,t,r> in RequirementSet
                , z in ord(RequirementSet,<c,d,t,r>) .. ord(RequirementSet,<c,d,t,r>)
                , i in 1..r
};
//
// decision variables
//
dvar int Start[InstanceSet] in Time;               // the course starting point
dvar int room[InstanceSet] in RoomId;              // the room in which the course is held
dvar int teacher[InstanceSet] in TeacherId;        // the teacher in charge of the course
//
// helper variables
//

dvar int End[InstanceSet] in Time;                    // the course end time
dvar int classTeacher[Class,Discipline] in TeacherId; // teacher working once per time point
dvar int makespan in Time;                            // ending date of last course
//
// search setup
//

execute {
   writeln("MaxTime = ", MaxTime);
   writeln("DayDuration = ", DayDuration);
   writeln("Teacher = ", Teacher);
   writeln("Discipline = ", Discipline);
   writeln("Class = ", Class);
   var p = cp.param;
   p.logPeriod = 10000;
   p.timeLimit = 600;
}

// minimize makespan
minimize makespan;

subject to { 
  makespan == max(r in InstanceSet) End[r];
  // help proving optimality
  makespan >= max(c in Class) sum(r in InstanceSet : r.Class == c) r.Duration;
  // ensure the course ends after it starts
  forall(r in InstanceSet)
    End[r] == r.Duration + Start[r];
  // ensure course numerotation is chronological
  forall(i, j in InstanceSet 
         : i.id < j.id 
           && i.requirementId == j.requirementId) 
    Start[i] < Start[j];
  // ensure that a teacher is required once at any time point.
  forall(r in InstanceSet, x in Teacher) {
    if(r.discipline in PossibleTeacherDiscipline[x])
      (sum(o in InstanceSet
                                : r.discipline in PossibleTeacherDiscipline[x])
        (Start[o] >= Start[r])
        *(Start[o] < End[r])
        *(teacher[o] == ord(Teacher,x))) < 2;
  }
  // ensure the teacher can teach the discipline
  forall(r in InstanceSet) 
    teacher[r] in PossibleTeacherIds[r.discipline];
     
  // ensure that a room is required once at any time point.
  forall(r in InstanceSet, x in Room) {
    if(PossibleRoom[r.discipline,x] == 1)
      (sum(o in InstanceSet : 1 == PossibleRoom[o.discipline,x])
        (Start[o] >= Start[r])
        *(Start[o] < End[r])
        *(room[o] == ord(Room,x))) < 2;            
  } 
  // ensure the room can support the discipline
  forall(r in InstanceSet)
    room[r] in PossibleRoomIds[r.discipline];
  // ensure that a class follows one course at a time
  forall(r in InstanceSet, x in Class) {
    if(r.Class == x)
      (sum(o in InstanceSet : o.Class == x) 
       (1 == (Start[o] >= Start[r])*(Start[o] < End[r]))) < 2;
  }
  // ensure that for given class and discipline, the teacher is always the same
  forall(c in Class, d in Discipline, r in InstanceSet 
         : r.Class == c && r.discipline == d) 
    teacher[r] == classTeacher[c, d];
   
  // ensure a course starts and end the same halfday
  forall(i in InstanceSet : i.Duration > 1)
    (Start[i] div HalfDayDuration) == ((End[i]-1) div HalfDayDuration);
  // insert break duration between specified disciplines
  forall(ordered i, j in InstanceSet, a,b in Discipline
         : (<b,a> in NeedBreak || <a,b> in NeedBreak)
         && i != j
         && i.Class == j.Class
         && ((i.discipline == a && j.discipline == b)
             || (i.discipline == b && j.discipline == a)))
    // courses do not belong to the same day
    ((Start[i] div DayDuration) != (Start[j] div DayDuration)) ||
    // courses do not belong to the same halfday
    ((Start[i] div HalfDayDuration) != (Start[j] div HalfDayDuration)) ||
    // courses are separated by BreakDuration
    ((Start[i] > End[j])*(Start[i] - End[j]) + 
     (Start[j] > End[i])*(Start[j] - End[i])) >= BreakDuration;
  // avoid to have the same discipline taught twice a day
  forall(ordered i,j in InstanceSet: i.discipline == j.discipline && i.Class == j.Class) 
    (Start[i] div DayDuration) != (Start[j] div DayDuration);
  // ensure that the morning disciplines end in the morning
  forall(d in MorningDiscipline, i in InstanceSet
         : i.discipline == d) 
    (Start[i] % DayDuration) < HalfDayDuration;

  // ===== BỔ SUNG cho bài 3.2 — ràng buộc khả dụng =====
  // Một instance r chiếm khoảng thời gian [Start[r], End[r]). Nó KHÔNG phủ thời
  // điểm t khi và chỉ khi Start[r] > t hoặc End[r] <= t.

  // RB7 — giáo viên bận. teacher[r] là BIẾN nên phải viết dạng kéo theo:
  // chỉ khi r được giao cho đúng giáo viên đang bận thì mới cấm.
  forall(r in InstanceSet, b in TeacherBusySet)
    (teacher[r] == ord(Teacher, b.who)) =>
      (Start[r] > b.t || End[r] <= b.t);

  // RB8 — lớp bận. r.Class là HẰNG của instance nên lọc thẳng ở vế điều kiện,
  // rẻ hơn hẳn RB7: không sinh ràng buộc kéo theo nào.
  forall(r in InstanceSet, b in ClassBusySet : r.Class == b.who)
    Start[r] > b.t || End[r] <= b.t;

  // RB4 — khung giờ cần tránh của môn. Cùng dạng RB8.
  forall(r in InstanceSet, b in DisciplineAvoidSet : r.discipline == b.who)
    Start[r] > b.t || End[r] <= b.t;
  // ===== hết phần bổ sung =====
};

//
// generate time table
//
tuple Course {
   string teacher;
   string discipline;
   string room;
   int    id;
   int    repetition;
};

{Course} timetable[t in Time][c in Class] = {
  <p,d,r,i,n> 
  | d in Discipline
  , r in Room
  , x in InstanceSet
  , n in x.repetition..x.repetition
  , p in Teacher 
  , i in x.id..x.id
  : (t >= Start[x])
  && (t < End[x])
  && (x.Class == c)
  && (room[x] == ord(Room,r))
  && (ord(Teacher,p) == teacher[x])
  && (d == x.discipline) 
};
   
// force execution of postprocessing expressions
execute POST_PROCESS {
  timetable;
  for(var c in Class) {
    writeln("Class ", c);
    var day = 0;
    for(var t = 0; t < makespan; t++) {
      if(t % DayDuration == 0) {
        day++;
        writeln("Day ", day);
      }
      if(t % DayDuration == HalfDayDuration) 
        writeln("Lunch break");
      var activity = 0;
      for(var x in timetable[t][c]) {
        activity++;
        writeln((t % DayDuration)+1, "\t",
                x.room, "\t", 
                x.discipline, "\t", 
                x.id, "/", 
                x.repetition, "\t", 
                x.teacher);
      }
      if(activity == 0)
        writeln((t % DayDuration)+1, "\tFree time");
    }
  }
}
 

// ===== BỔ SUNG — dòng RESULT theo giao kèo của tools/runner.py =====
execute EMIT_RESULT {
  writeln("RESULT {\"status\":\"Optimal\""
        + ",\"objective\":" + makespan
        + ",\"solve_time_s\":" + cp.info.solveTime
        + ",\"branches\":" + cp.info.numberOfBranches
        + ",\"fails\":" + cp.info.numberOfFails
        + ",\"nb_teacher_busy\":" + TeacherBusySet.size
        + ",\"nb_class_busy\":" + ClassBusySet.size
        + ",\"nb_discipline_avoid\":" + DisciplineAvoidSet.size + "}");
}
```

### DOcplex.cp → engine CP Optimizer

In [13]:
show_dimension_code("3.2_timetable", "docplexcp")

**Python (docplex.cp) → engine CP Optimizer · ✍️ viết mới**

```python
"""Bài 3.2 — Thời khoá biểu có ràng buộc khả dụng | Chiều DOcplex.cp (engine CP Optimizer)

Nguồn: ✍️ VIẾT MỚI. IBM không phát hành bản CP nào của bài xếp thời khoá biểu qua
`docplex.cp` (chỉ có `timetabling.mod` bằng OPL). Bản này dựng lại đúng mô hình đó
bằng Python, cộng ba nhóm ràng buộc khả dụng RB7/RB8/RB4.

Đọc ĐÚNG những file `.dat` gốc mà hai chiều kia dùng, qua `tools/opl_dat.py`.

Chạy:
    python3 models/3.2_timetable/docplexcp/timetable_cp.py \\
        data/timetable/base.dat data/timetable/large.dat data/timetable/availability.dat

VAI TRÒ CỦA CHIỀU NÀY TRONG BÁO CÁO
------------------------------------
Đây là điểm đo thứ ba, và là điểm đo duy nhất tách bạch được hai nguyên nhân:

    OPL (CP Optimizer + đếm có điều kiện)
      ──vs── bản này (CP Optimizer + interval)      → cô lập ảnh hưởng của MÃ HOÁ
      ──vs── OR-Tools (CP-SAT + interval)           → cô lập ảnh hưởng của ENGINE

So thẳng OPL với OR-Tools thì hai yếu tố đổi cùng lúc nên không kết luận được gì.

HAI PRIMITIVE CỦA CP OPTIMIZER MÀ CP-SAT KHÔNG CÓ
--------------------------------------------------
1. `alternative(master, [alt1, alt2, ...])` — chọn đúng một tài nguyên trong nhiều
   lựa chọn, tự đồng bộ thời gian với interval chính. Bản CP-SAT phải làm tay:
   tạo literal hiện diện cho từng cặp (khoá học, tài nguyên) rồi tự buộc chúng
   khớp với biến chọn tài nguyên.

2. `forbid_extent(interval, F)` + hàm bậc thang — cấm một interval phủ lên vùng mà
   F bằng 0. Đây là cách diễn đạt LỊCH BẬN đúng nghĩa: khai báo trực tiếp lịch của
   tài nguyên, không cần biến phụ nào. Bản CP-SAT phải bung thành phép tuyển
   "bắt đầu sau t HOẶC kết thúc trước t" kèm một biến bool cho mỗi cặp
   (khoá học, thời điểm bận).

Ba nhóm ràng buộc khả dụng ở đây vì thế **không sinh thêm một biến quyết định nào**
— chúng chỉ là dữ liệu lịch gắn vào interval. Đây là thế mạnh 20 năm chuyên biệt
của CP Optimizer cho bài lập lịch mà brief §1 đã nêu.
"""

from __future__ import annotations

import collections
import json
import pathlib
import sys
import time

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[3] / "tools"))
import cpo_env  # noqa: F401,E402  — trỏ docplex.cp sang cpoptimizer cục bộ
from opl_dat import load  # noqa: E402

from docplex.cp.function import CpoStepFunction  # noqa: E402
from docplex.cp.model import CpoModel  # noqa: E402

Instance = collections.namedtuple(
    "Instance", "cls discipline duration repetition id requirement_id"
)


def _blocking_function(busy_times, horizon: int) -> CpoStepFunction:
    """Hàm bậc thang: 1 ở mọi thời điểm rảnh, 0 tại các thời điểm bận.

    Dùng với forbid_extent để cấm interval phủ lên vùng bằng 0.
    """
    fn = CpoStepFunction()
    fn.set_value(0, horizon, 1)
    for t in busy_times:
        fn.set_value(t, t + 1, 0)
    return fn


def build_and_solve(dat_files: list[str], time_limit: float = 300.0,
                    workers: int = 8, seed: int = 0) -> dict:
    d = load(*dat_files)

    requirements = d["RequirementSet"]
    teacher_discipline = d["TeacherDisciplineSet"]
    rooms = d["Room"]
    dedicated = set(d["DedicatedRoomSet"])
    need_break = set(d.get("NeedBreak", []))
    morning = set(d.get("MorningDiscipline", []))
    break_duration = d["BreakDuration"]
    day_duration = d["DayDuration"]
    nb_days = d["NumberOfDaysPerPeriod"]

    classes = sorted({r[0] for r in requirements})
    teachers = sorted({t for t, _ in teacher_discipline})
    disciplines = sorted({s for _, s in teacher_discipline})

    half_day = day_duration // 2
    max_time = day_duration * nb_days
    teacher_id = {t: i for i, t in enumerate(teachers)}
    room_id = {r: i for i, r in enumerate(rooms)}

    can_teach: dict[str, list[int]] = {s: [] for s in disciplines}
    for t, s in teacher_discipline:
        can_teach[s].append(teacher_id[t])

    dedicated_rooms = {x for x, _ in dedicated}
    disciplines_needing_room = {s for _, s in dedicated}
    can_use_room = {
        s: [
            room_id[x]
            for x in rooms
            if (x, s) in dedicated
            or (x not in dedicated_rooms and s not in disciplines_needing_room)
        ]
        for s in disciplines
    }

    instances: list[Instance] = []
    for req_idx, (cls, disc, dur, rep) in enumerate(requirements):
        for i in range(1, rep + 1):
            instances.append(Instance(cls, disc, dur, rep, i, req_idx))
    n = len(instances)

    mdl = CpoModel()

    # ---- interval chính cho mỗi khoá học ------------------------------------
    # end cũng bị chặn ở max_time-1, đúng như bản OPL gốc khai End thuộc Time.
    x = [
        mdl.interval_var(
            size=inst.duration,
            start=(0, max_time - 1),
            end=(0, max_time - 1),
            name=f"x{i}_{inst.cls}_{inst.discipline}",
        )
        for i, inst in enumerate(instances)
    ]

    # ---- chọn giáo viên và phòng bằng alternative ---------------------------
    # alternative(master, alts) = đúng một alt hiện diện và đồng bộ thời gian với
    # master. Bản CP-SAT phải dựng tay bằng literal hiện diện.
    xt: list[dict[int, object]] = []
    xr: list[dict[int, object]] = []
    for i, inst in enumerate(instances):
        alts_t = {
            tid: mdl.interval_var(
                size=inst.duration, start=(0, max_time - 1), end=(0, max_time - 1),
                optional=True, name=f"xt{i}_{tid}",
            )
            for tid in can_teach[inst.discipline]
        }
        mdl.add(mdl.alternative(x[i], list(alts_t.values())))
        xt.append(alts_t)

        alts_r = {
            rid: mdl.interval_var(
                size=inst.duration, start=(0, max_time - 1), end=(0, max_time - 1),
                optional=True, name=f"xr{i}_{rid}",
            )
            for rid in can_use_room[inst.discipline]
        }
        mdl.add(mdl.alternative(x[i], list(alts_r.values())))
        xr.append(alts_r)

    # ---- tài nguyên dùng một lần tại mỗi thời điểm --------------------------
    for tid in range(len(teachers)):
        ivs = [xt[i][tid] for i in range(n) if tid in xt[i]]
        if ivs:
            mdl.add(mdl.no_overlap(ivs))
    for rid in range(len(rooms)):
        ivs = [xr[i][rid] for i in range(n) if rid in xr[i]]
        if ivs:
            mdl.add(mdl.no_overlap(ivs))

    by_class: dict[str, list[int]] = collections.defaultdict(list)
    for i, inst in enumerate(instances):
        by_class[inst.cls].append(i)
    for idxs in by_class.values():
        mdl.add(mdl.no_overlap([x[i] for i in idxs]))

    # ---- một lớp học một môn thì luôn cùng một giáo viên --------------------
    by_class_discipline: dict[tuple[str, str], list[int]] = collections.defaultdict(list)
    for i, inst in enumerate(instances):
        by_class_discipline[(inst.cls, inst.discipline)].append(i)
    for (_, disc), idxs in by_class_discipline.items():
        head = idxs[0]
        for i in idxs[1:]:
            for tid in can_teach[disc]:
                mdl.add(mdl.presence_of(xt[i][tid]) == mdl.presence_of(xt[head][tid]))

    # ---- thứ tự thời gian giữa các buổi của cùng một requirement ------------
    for i in range(n):
        for j in range(n):
            if (
                instances[i].requirement_id == instances[j].requirement_id
                and instances[i].id < instances[j].id
            ):
                mdl.add(mdl.start_of(x[i]) < mdl.start_of(x[j]))

    # ---- ràng buộc lịch, diễn đạt bằng hàm bậc thang ------------------------
    # môn buổi sáng: chỉ được BẮT ĐẦU trong buổi sáng
    morning_fn = CpoStepFunction()
    morning_fn.set_value(0, max_time, 0)
    for day in range(nb_days):
        morning_fn.set_value(day * day_duration, day * day_duration + half_day, 1)

    # khoá dài hơn 1 tiết phải nằm gọn trong một buổi
    same_halfday_fn: dict[int, CpoStepFunction] = {}
    for dur in {inst.duration for inst in instances if inst.duration > 1}:
        fn = CpoStepFunction()
        fn.set_value(0, max_time, 0)
        for s in range(max_time):
            if (s % half_day) + dur <= half_day:
                fn.set_value(s, s + 1, 1)
        same_halfday_fn[dur] = fn

    for i, inst in enumerate(instances):
        if inst.discipline in morning:
            mdl.add(mdl.forbid_start(x[i], morning_fn))
        if inst.duration > 1:
            mdl.add(mdl.forbid_start(x[i], same_halfday_fn[inst.duration]))

    # ---- không dạy trùng môn hai lần trong một ngày, với cùng một lớp -------
    for i in range(n):
        for j in range(i + 1, n):
            if (
                instances[i].discipline == instances[j].discipline
                and instances[i].cls == instances[j].cls
            ):
                mdl.add(
                    mdl.int_div(mdl.start_of(x[i]), day_duration)
                    != mdl.int_div(mdl.start_of(x[j]), day_duration)
                )

    # ---- giờ nghỉ giữa hai môn kỵ nhau --------------------------------------
    for i in range(n):
        for j in range(i + 1, n):
            a, b = instances[i].discipline, instances[j].discipline
            if instances[i].cls != instances[j].cls:
                continue
            if (a, b) not in need_break and (b, a) not in need_break:
                continue
            si, ei = mdl.start_of(x[i]), mdl.end_of(x[i])
            sj, ej = mdl.start_of(x[j]), mdl.end_of(x[j])
            gap = mdl.max([0, si - ej]) + mdl.max([0, sj - ei])
            mdl.add(
                (mdl.int_div(si, day_duration) != mdl.int_div(sj, day_duration))
                | (mdl.int_div(si, half_day) != mdl.int_div(sj, half_day))
                | (gap >= break_duration)
            )

    # ---- RB7 / RB8 / RB4 — ràng buộc khả dụng ------------------------------
    # Khai báo trực tiếp lịch bận của tài nguyên rồi cấm interval phủ lên đó.
    # Không sinh thêm biến quyết định nào — khác hẳn bản CP-SAT.
    teacher_busy: dict[str, list[int]] = collections.defaultdict(list)
    for who, t in d.get("TeacherBusySet", []):
        teacher_busy[who].append(t)
    for who, times in teacher_busy.items():
        tid = teacher_id.get(who)
        if tid is None:
            continue
        fn = _blocking_function(times, max_time)
        for i in range(n):
            if tid in xt[i]:
                mdl.add(mdl.forbid_extent(xt[i][tid], fn))

    class_busy: dict[str, list[int]] = collections.defaultdict(list)
    for who, t in d.get("ClassBusySet", []):
        class_busy[who].append(t)
    for who, times in class_busy.items():
        fn = _blocking_function(times, max_time)
        for i, inst in enumerate(instances):
            if inst.cls == who:
                mdl.add(mdl.forbid_extent(x[i], fn))

    discipline_avoid: dict[str, list[int]] = collections.defaultdict(list)
    for who, t in d.get("DisciplineAvoidSet", []):
        discipline_avoid[who].append(t)
    for who, times in discipline_avoid.items():
        fn = _blocking_function(times, max_time)
        for i, inst in enumerate(instances):
            if inst.discipline == who:
                mdl.add(mdl.forbid_extent(x[i], fn))

    # ---- mục tiêu -----------------------------------------------------------
    load_per_class = collections.Counter()
    for inst in instances:
        load_per_class[inst.cls] += inst.duration
    makespan = mdl.integer_var(0, max_time - 1, "makespan")
    mdl.add(makespan == mdl.max([mdl.end_of(x[i]) for i in range(n)]))
    mdl.add(makespan >= max(load_per_class.values()))
    mdl.add(mdl.minimize(makespan))

    t0 = time.perf_counter()
    sol = mdl.solve(TimeLimit=time_limit, Workers=workers, RandomSeed=seed,
                    LogVerbosity="Quiet")
    wall = time.perf_counter() - t0

    infos = sol.get_solver_infos() if sol else {}
    result = {
        "status": str(sol.get_solve_status()) if sol else "NoSolution",
        "objective": sol.get_objective_value() if sol else None,
        "solve_time_s": round(sol.get_solve_time(), 4) if sol else None,
        "branches": infos.get("NumberOfBranches"),
        "fails": infos.get("NumberOfFails"),
        "nb_instances": n,
        "nb_teacher_busy": len(d.get("TeacherBusySet", [])),
        "nb_class_busy": len(d.get("ClassBusySet", [])),
        "nb_discipline_avoid": len(d.get("DisciplineAvoidSet", [])),
        "wall_s": round(wall, 4),
    }

    if sol:
        print(f"Makespan = {result['objective']}  ({n} khoá học)")
        for c in classes:
            print(f"\n=== Lớp {c} ===")
            for s_val, i in sorted((sol.get_var_solution(x[i]).get_start(), i)
                                   for i in by_class[c]):
                inst = instances[i]
                who = next(teachers[t] for t in xt[i]
                           if sol.get_var_solution(xt[i][t]).is_present())
                where = next(rooms[r] for r in xr[i]
                             if sol.get_var_solution(xr[i][r]).is_present())
                print(
                    f"  ngày {s_val // day_duration + 1} tiết {s_val % day_duration + 1}"
                    f"  {inst.discipline:<10} {who:<14} phòng {where}"
                )
    return result


if __name__ == "__main__":
    files = [a for a in sys.argv[1:] if a.endswith(".dat")]
    if not files:
        files = ["data/timetable/base.dat", "data/timetable/large.dat",
                 "data/timetable/availability.dat"]
    print("RESULT " + json.dumps(build_and_solve(files)))
```

### OR-Tools → engine CP-SAT

In [14]:
show_dimension_code("3.2_timetable", "ortools")

**Python (ortools.sat) → engine CP-SAT · ✍️ viết mới**

```python
"""Bài 3.2 — Thời khoá biểu có ràng buộc khả dụng | Chiều OR-Tools (engine CP-SAT)

Nguồn: ✍️ VIẾT MỚI. Port đầy đủ mô hình `timetabling.mod` của IBM (CPLEX Studio
22.2) sang CP-SAT, cộng ba nhóm ràng buộc khả dụng RB7/RB8/RB4 mà bản gốc không có.

Đọc ĐÚNG những file `.dat` gốc mà chiều OPL dùng, qua `tools/opl_dat.py` — ba
chiều không có bản dữ liệu riêng nào để lệch nhau.

Chạy:
    python3 models/3.2_timetable/ortools/timetable_sat.py \\
        data/timetable/base.dat data/timetable/large.dat data/timetable/availability.dat

    # bỏ availability để đo chi phí của nó:
    python3 models/3.2_timetable/ortools/timetable_sat.py \\
        data/timetable/base.dat data/timetable/large.dat

KHÁC BIỆT MÃ HOÁ SO VỚI BẢN OPL (điểm so sánh cốt lõi của bài này)
------------------------------------------------------------------
Bản OPL gốc diễn đạt "tài nguyên chỉ dùng một lần tại mỗi thời điểm" bằng TỔNG CÓ
ĐIỀU KIỆN — với mỗi khoá học r nó đếm xem có bao nhiêu khoá khác bắt đầu trong
khoảng [Start[r], End[r]) rồi bắt tổng đó < 2:

    (sum(o in InstanceSet : o.Class == x)
       (1 == (Start[o] >= Start[r])*(Start[o] < End[r]))) < 2;

Bản này dùng BIẾN INTERVAL + no-overlap, thứ cả CP Optimizer lẫn CP-SAT đều có
sẵn thuật toán lan truyền chuyên dụng:

    model.add_no_overlap([...các interval của cùng một lớp...])

Hai cách tương đương về nghiệm (hai đoạn giao nhau thì điểm bắt đầu muộn hơn luôn
nằm trong khoảng của đoạn kia), nhưng khác hẳn về sức lan truyền. Với tài nguyên
được chọn bằng BIẾN (giáo viên, phòng), ở đây dùng interval TUỲ CHỌN gắn literal
hiện diện — đúng cách mà một solver lập lịch mong nhận được bài toán.
"""

from __future__ import annotations

import collections
import json
import pathlib
import sys
import time

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[3] / "tools"))
from opl_dat import load  # noqa: E402

from ortools.sat.python import cp_model  # noqa: E402

Instance = collections.namedtuple(
    "Instance", "cls discipline duration repetition id requirement_id"
)


def build_and_solve(dat_files: list[str], time_limit: float = 300.0, workers: int = 8,
                    seed: int = 0) -> dict:
    d = load(*dat_files)

    # ---- từ vựng, dựng đúng như phần đầu timetabling.mod ---------------------
    requirements = d["RequirementSet"]
    teacher_discipline = d["TeacherDisciplineSet"]
    rooms = d["Room"]
    dedicated = set(d["DedicatedRoomSet"])
    need_break = set(d.get("NeedBreak", []))
    morning = set(d.get("MorningDiscipline", []))
    break_duration = d["BreakDuration"]
    day_duration = d["DayDuration"]
    nb_days = d["NumberOfDaysPerPeriod"]

    classes = sorted({r[0] for r in requirements})
    teachers = sorted({t for t, _ in teacher_discipline})
    disciplines = sorted({s for _, s in teacher_discipline})

    half_day = day_duration // 2
    max_time = day_duration * nb_days

    teacher_id = {t: i for i, t in enumerate(teachers)}
    room_id = {r: i for i, r in enumerate(rooms)}

    # Giáo viên nào dạy được môn nào.
    can_teach: dict[str, list[int]] = {s: [] for s in disciplines}
    for t, s in teacher_discipline:
        can_teach[s].append(teacher_id[t])

    # Phòng nào dùng được cho môn nào — công thức PossibleRoom của bản gốc:
    # phòng x dùng được cho môn s nếu <x,s> là phòng chuyên dụng, HOẶC nếu x không
    # chuyên dụng cho môn nào và s cũng không đòi phòng chuyên dụng nào.
    dedicated_rooms = {x for x, _ in dedicated}
    disciplines_needing_room = {s for _, s in dedicated}
    can_use_room: dict[str, list[int]] = {}
    for s in disciplines:
        ok = [
            room_id[x]
            for x in rooms
            if (x, s) in dedicated
            or (x not in dedicated_rooms and s not in disciplines_needing_room)
        ]
        can_use_room[s] = ok

    # Mỗi requirement lặp `repetition` lần, mỗi lần là một instance.
    instances: list[Instance] = []
    for req_idx, (cls, disc, dur, rep) in enumerate(requirements):
        for i in range(1, rep + 1):
            instances.append(Instance(cls, disc, dur, rep, i, req_idx))
    n = len(instances)

    # ---- biến quyết định ----------------------------------------------------
    model = cp_model.CpModel()
    dom = cp_model.Domain(0, max_time - 1)

    start = [model.new_int_var(0, max_time - 1, f"start{i}") for i in range(n)]
    # Bản gốc khai End cũng thuộc Time nên khoá học phải kết thúc trước MaxTime-1.
    end = [model.new_int_var_from_domain(dom, f"end{i}") for i in range(n)]
    teacher = [
        model.new_int_var_from_domain(
            cp_model.Domain.from_values(can_teach[inst.discipline]), f"teacher{i}"
        )
        for i, inst in enumerate(instances)
    ]
    room = [
        model.new_int_var_from_domain(
            cp_model.Domain.from_values(can_use_room[inst.discipline]), f"room{i}"
        )
        for i, inst in enumerate(instances)
    ]
    makespan = model.new_int_var(0, max_time - 1, "makespan")

    # classTeacher[c,s]: một lớp học một môn thì luôn cùng một giáo viên.
    class_teacher = {}
    for c in classes:
        for s in disciplines:
            class_teacher[(c, s)] = model.new_int_var(
                0, len(teachers) - 1, f"classTeacher_{c}_{s}"
            )

    for i, inst in enumerate(instances):
        model.add(end[i] == start[i] + inst.duration)
        model.add(teacher[i] == class_teacher[(inst.cls, inst.discipline)])

    # ---- makespan -----------------------------------------------------------
    model.add_max_equality(makespan, end)
    load_per_class = collections.Counter()
    for inst in instances:
        load_per_class[inst.cls] += inst.duration
    model.add(makespan >= max(load_per_class.values()))   # chặn dưới, giúp chứng minh tối ưu

    # ---- thứ tự thời gian giữa các buổi của cùng một requirement -------------
    for i in range(n):
        for j in range(n):
            if (
                instances[i].requirement_id == instances[j].requirement_id
                and instances[i].id < instances[j].id
            ):
                model.add(start[i] < start[j])

    # ---- lớp học một môn tại một thời điểm: interval thường + no-overlap -----
    by_class: dict[str, list[int]] = collections.defaultdict(list)
    for i, inst in enumerate(instances):
        by_class[inst.cls].append(i)
    for c, idxs in by_class.items():
        model.add_no_overlap(
            [
                model.new_interval_var(start[i], instances[i].duration, end[i], f"iv_c{c}_{i}")
                for i in idxs
            ]
        )

    # ---- giáo viên và phòng: interval TUỲ CHỌN + no-overlap -----------------
    # Tài nguyên ở đây do BIẾN chọn, nên mỗi cặp (khoá học, tài nguyên) có một
    # interval chỉ "hiện diện" khi biến chọn đúng tài nguyên đó.
    teacher_lit: dict[tuple[int, int], object] = {}
    teacher_intervals: dict[int, list] = collections.defaultdict(list)
    for i, inst in enumerate(instances):
        for tid in can_teach[inst.discipline]:
            lit = model.new_bool_var(f"t_{i}_{tid}")
            model.add(teacher[i] == tid).only_enforce_if(lit)
            model.add(teacher[i] != tid).only_enforce_if(~lit)
            teacher_lit[(i, tid)] = lit
            teacher_intervals[tid].append(
                model.new_optional_interval_var(
                    start[i], inst.duration, end[i], lit, f"iv_t{tid}_{i}"
                )
            )
    for tid, ivs in teacher_intervals.items():
        model.add_no_overlap(ivs)

    room_intervals: dict[int, list] = collections.defaultdict(list)
    for i, inst in enumerate(instances):
        for rid in can_use_room[inst.discipline]:
            lit = model.new_bool_var(f"r_{i}_{rid}")
            model.add(room[i] == rid).only_enforce_if(lit)
            model.add(room[i] != rid).only_enforce_if(~lit)
            room_intervals[rid].append(
                model.new_optional_interval_var(
                    start[i], inst.duration, end[i], lit, f"iv_r{rid}_{i}"
                )
            )
    for rid, ivs in room_intervals.items():
        model.add_no_overlap(ivs)

    # ---- chỉ số ngày / buổi, cần cho các ràng buộc lịch ---------------------
    day = [model.new_int_var(0, nb_days - 1, f"day{i}") for i in range(n)]
    hd_start = [model.new_int_var(0, 2 * nb_days - 1, f"hds{i}") for i in range(n)]
    hd_end = [model.new_int_var(0, 2 * nb_days - 1, f"hde{i}") for i in range(n)]
    slot = [model.new_int_var(0, day_duration - 1, f"slot{i}") for i in range(n)]
    for i in range(n):
        model.add_division_equality(day[i], start[i], day_duration)
        model.add_division_equality(hd_start[i], start[i], half_day)
        model.add_modulo_equality(slot[i], start[i], day_duration)
        end_minus_1 = model.new_int_var(0, max_time - 1, f"em1_{i}")
        model.add(end_minus_1 == end[i] - 1)
        model.add_division_equality(hd_end[i], end_minus_1, half_day)

    # khoá học phải bắt đầu và kết thúc trong cùng một buổi
    for i, inst in enumerate(instances):
        if inst.duration > 1:
            model.add(hd_start[i] == hd_end[i])

    # môn buổi sáng phải nằm trong buổi sáng
    for i, inst in enumerate(instances):
        if inst.discipline in morning:
            model.add(slot[i] < half_day)

    # không dạy trùng môn hai lần trong một ngày, với cùng một lớp
    for i in range(n):
        for j in range(i + 1, n):
            if (
                instances[i].discipline == instances[j].discipline
                and instances[i].cls == instances[j].cls
            ):
                model.add(day[i] != day[j])

    # ---- giờ nghỉ giữa hai môn kỵ nhau --------------------------------------
    for i in range(n):
        for j in range(i + 1, n):
            a, b = instances[i].discipline, instances[j].discipline
            if instances[i].cls != instances[j].cls:
                continue
            if (a, b) not in need_break and (b, a) not in need_break:
                continue
            diff_day = model.new_bool_var(f"bd_{i}_{j}")
            model.add(day[i] != day[j]).only_enforce_if(diff_day)
            model.add(day[i] == day[j]).only_enforce_if(~diff_day)

            diff_hd = model.new_bool_var(f"bh_{i}_{j}")
            model.add(hd_start[i] != hd_start[j]).only_enforce_if(diff_hd)
            model.add(hd_start[i] == hd_start[j]).only_enforce_if(~diff_hd)

            # khoảng cách giữa hai khoá; hai khoá cùng lớp không chồng nhau nên
            # nhiều nhất một trong hai số hạng dương.
            g1 = model.new_int_var(0, max_time, f"g1_{i}_{j}")
            g2 = model.new_int_var(0, max_time, f"g2_{i}_{j}")
            model.add_max_equality(g1, [0, start[i] - end[j]])
            model.add_max_equality(g2, [0, start[j] - end[i]])
            far = model.new_bool_var(f"bg_{i}_{j}")
            model.add(g1 + g2 >= break_duration).only_enforce_if(far)
            model.add(g1 + g2 < break_duration).only_enforce_if(~far)

            model.add_bool_or([diff_day, diff_hd, far])

    # ---- RB7 / RB8 / RB4 — ràng buộc khả dụng (phần bài 3.2 thêm vào) -------
    # Khoá học i chiếm [start, end). Nó KHÔNG phủ thời điểm t khi start > t hoặc
    # end <= t. Biến `after` chọn một trong hai vế của phép tuyển đó.
    def forbid(i: int, t: int, tag: str, extra_lit=None) -> None:
        after = model.new_bool_var(f"av_{tag}_{i}_{t}")
        guard = [after] if extra_lit is None else [extra_lit, after]
        model.add(start[i] > t).only_enforce_if(guard)
        guard = [~after] if extra_lit is None else [extra_lit, ~after]
        model.add(end[i] <= t).only_enforce_if(guard)

    # RB7 — giáo viên bận. Giáo viên là BIẾN nên phải gắn thêm literal hiện diện.
    for who, t in d.get("TeacherBusySet", []):
        tid = teacher_id.get(who)
        if tid is None:
            continue
        for i, inst in enumerate(instances):
            if tid in can_teach[inst.discipline]:
                forbid(i, t, "t", teacher_lit[(i, tid)])

    # RB8 — lớp bận. Lớp là HẰNG của instance nên lọc thẳng, không cần literal.
    for who, t in d.get("ClassBusySet", []):
        for i, inst in enumerate(instances):
            if inst.cls == who:
                forbid(i, t, "c")

    # RB4 — khung giờ cần tránh của môn.
    for who, t in d.get("DisciplineAvoidSet", []):
        for i, inst in enumerate(instances):
            if inst.discipline == who:
                forbid(i, t, "d")

    # ---- giải ---------------------------------------------------------------
    model.minimize(makespan)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    # Cố định để số liệu benchmark tái lập được — xem PLAN.md §2.4.
    solver.parameters.num_workers = workers
    solver.parameters.random_seed = seed
    t0 = time.perf_counter()
    status = solver.solve(model)
    wall = time.perf_counter() - t0

    result = {
        "status": solver.status_name(status),
        "objective": solver.value(makespan) if status in (cp_model.OPTIMAL, cp_model.FEASIBLE) else None,
        "solve_time_s": round(solver.wall_time, 4),
        "branches": solver.num_branches,
        "fails": solver.num_conflicts,      # CP-SAT đếm conflicts, không phải fails
        "nb_instances": n,
        "nb_teacher_busy": len(d.get("TeacherBusySet", [])),
        "nb_class_busy": len(d.get("ClassBusySet", [])),
        "nb_discipline_avoid": len(d.get("DisciplineAvoidSet", [])),
        "wall_s": round(wall, 4),
    }

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(f"Makespan = {solver.value(makespan)}  ({n} khoá học)")
        for c in classes:
            print(f"\n=== Lớp {c} ===")
            rows = sorted(
                (solver.value(start[i]), i) for i in by_class[c]
            )
            for s_val, i in rows:
                inst = instances[i]
                print(
                    f"  ngày {s_val // day_duration + 1} tiết {s_val % day_duration + 1}"
                    f"  {inst.discipline:<10} {teachers[solver.value(teacher[i])]:<14}"
                    f" phòng {rooms[solver.value(room[i])]}"
                )
    return result


if __name__ == "__main__":
    files = [a for a in sys.argv[1:] if a.endswith(".dat")]
    if not files:
        files = ["data/timetable/base.dat", "data/timetable/large.dat",
                 "data/timetable/availability.dat"]
    print("RESULT " + json.dumps(build_and_solve(files)))
```

## Chạy — cả ba chiều, cùng một bộ dữ liệu

Bảng dưới sinh ra bằng cách **chạy thật** ba file code vừa in ở trên.

In [15]:
run_table("3.2_timetable")

,chiều,ngôn ngữ,engine,nguồn,trạng thái,mục tiêu,thời gian giải (s),nhánh,fails/conflicts
0,opl,OPL,CP Optimizer,✅+✍️,Optimal,47,7.0210,125151,55086
1,docplexcp,Python (docplex.cp),CP Optimizer,✍️,Optimal,47,1.3150,154818,59201
2,ortools,Python (ortools.sat),CP-SAT,✍️,OPTIMAL,47,1.8963,5168,141


### Kiểm chứng chéo

Ba chiều phải cùng ra một nghiệm tối ưu. Không có bước này thì mọi so sánh hiệu năng đều vô nghĩa — nhanh hơn mà giải sai bài thì không nói lên điều gì.

In [16]:
cross_check("3.2_timetable")

**Kiểm chứng chéo — bài 3.2_timetable**

`opl` = 47 · `docplexcp` = 47 · `ortools` = 47 → ✅ **KHỚP** — các chiều cùng một nghiệm tối ưu


## (d) Quan sát so sánh

In [17]:
show_notes("3.2_timetable", "Quan sát")

## (d) Quan sát cho phần so sánh

### Chi phí của tính khả dụng

| Bộ `large` | log₂ không gian | Makespan | OPL | DOcplex.cp | OR-Tools |
|---|---|---|---|---|---|
| không availability | 786.4 | 44 | 17.2 s | 0.58 s | 2.46 s |
| có availability | **780.3** | 47 | 7.7 s | 1.41 s | 2.04 s |

Hai điều đáng nói:

1. **Thêm ràng buộc làm không gian tìm kiếm GIẢM** (786.4 → 780.3), dù số ràng
   buộc gần gấp đôi (2 120 → 3 947). Ràng buộc khả dụng chỉ cắt miền của biến sẵn
   có chứ không thêm biến mới. Community Edition tính trần theo *không gian tìm
   kiếm* chứ không theo *số ràng buộc* — nên còn nhiều chỗ để thêm ràng buộc nữa.
2. **Cái giá của tính khả dụng nằm ở chất lượng lời giải, không ở thời gian giải.**
   Makespan xấu đi 44 → 47, còn thời gian giải thì tuỳ chiều: OPL *nhanh hơn* hẳn
   (17.2 → 7.7 s) vì miền bị cắt bớt, DOcplex.cp chậm lại, OR-Tools gần như không đổi.
   Ràng buộc thực tế hơn làm lịch dài ra, chứ không làm bài toán khó hơn về tính toán.

### Hiệu năng — bảng trung vị 3 lần chạy

Sinh bằng `make bench` → `results/bench.csv`. Seed và số worker cố định để tái lập.

| Cấu hình | Chiều | Engine | Mã hoá | Obj | Thời gian | Nhánh | Fails/Confl |
|---|---|---|---|---|---|---|---|
| small | OPL | CP Optimizer | đếm có điều kiện | 20 | 0.32 s | 30 412 | 12 917 |
| small | DOcplex.cp | CP Optimizer | interval | 20 | 0.45 s | 75 025 | 33 825 |
| small | OR-Tools | CP-SAT | interval | 20 | **0.07 s** | 575 | 6 |
| large | OPL | CP Optimizer | đếm có điều kiện | 44 | 17.17 s | 216 066 | 96 460 |
| large | DOcplex.cp | CP Optimizer | interval | 44 | **0.58 s** | 34 194 | 7 037 |
| large | OR-Tools | CP-SAT | interval | 44 | 2.46 s | 7 095 | 185 |
| large+avail | OPL | CP Optimizer | đếm có điều kiện | 47 | 7.74 s | 125 151 | 55 086 |
| large+avail | DOcplex.cp | CP Optimizer | interval | 47 | **1.41 s** | 154 818 | 59 201 |
| large+avail | OR-Tools | CP-SAT | interval | 47 | 2.04 s | 5 399 | 141 |

### Đọc bảng: hai phép so, mỗi phép đổi đúng một biến số

**Trục MÃ HOÁ** — OPL vs DOcplex.cp, *cùng engine CP Optimizer*:

| Cấu hình | đếm có điều kiện | interval | tỉ lệ |
|---|---|---|---|
| small | 0.32 s | 0.45 s | 0.7× (interval *chậm hơn*) |
| large | 17.17 s | 0.58 s | **29× nhanh hơn** |
| large+avail | 7.74 s | 1.41 s | **5.5× nhanh hơn** |

Ở bài nhỏ, mã hoá interval còn thua vì chi phí dựng mô hình lớn hơn phần lợi. Bài
càng lớn thì càng thắng đậm — vì cách đếm có điều kiện sinh ra số ràng buộc tăng
theo **bình phương** số khoá học, trong khi `no_overlap` chỉ là một ràng buộc toàn
cục cho mỗi tài nguyên.

Đáng chú ý: ở `large+avail`, DOcplex.cp duyệt **nhiều nhánh hơn** OPL (154 818 so
với 125 151) mà vẫn nhanh hơn 5.5×. **Số nhánh và thời gian không đi cùng nhau** —
mỗi nhánh của mã hoá đếm-có-điều-kiện đắt hơn hẳn vì phải lan truyền qua hàng nghìn
tích có điều kiện.

**Trục ENGINE** — DOcplex.cp vs OR-Tools, *cùng mã hoá interval*:

| Cấu hình | CP Optimizer | CP-SAT | nhánh CPO | nhánh CP-SAT |
|---|---|---|---|---|
| small | 0.45 s | **0.07 s** | 75 025 | 575 |
| large | **0.58 s** | 2.46 s | 34 194 | 7 095 |
| large+avail | **1.41 s** | 2.04 s | 154 818 | 5 399 |

Hai engine đi hai lối rõ rệt. CP-SAT duyệt **ít hơn 30–130 lần** số nhánh nhờ học
mệnh đề xung đột, nhưng mỗi nhánh đắt hơn nhiều. CP Optimizer duyệt ồ ạt với chi
phí mỗi nhánh rất rẻ. Ở bài nhỏ, cách của CP-SAT thắng tuyệt đối; ở hai bài lớn,
CP Optimizer về trước — đúng với nhận định ở brief §1 rằng CP Optimizer mạnh nhất
ở bài lập lịch, nơi nó có 20 năm tối ưu riêng cho biến interval.

> Ghi nhớ khi đọc số: `fails` của CP Optimizer và `conflicts` của CP-SAT đếm hai
> thứ khác nhau, chỉ so được trong cùng một engine.

### Hai primitive CP Optimizer có mà CP-SAT không có

Chỗ này thấy rõ nhất khi đặt hai file Python cạnh nhau — cùng ngôn ngữ, cùng cách
mã hoá, chỉ khác thư viện.

**1. Chọn tài nguyên.** CP Optimizer có `alternative(master, [alt...])`: đúng một
lựa chọn hiện diện và tự đồng bộ thời gian với interval chính.

```python
mdl.add(mdl.alternative(x[i], list(alts_t.values())))          # docplex.cp
```

CP-SAT không có, phải dựng tay literal hiện diện cho từng cặp rồi tự buộc khớp:

```python
lit = model.new_bool_var(f"t_{i}_{tid}")                        # ortools
model.add(teacher[i] == tid).only_enforce_if(lit)
model.add(teacher[i] != tid).only_enforce_if(~lit)
```

**2. Lịch bận.** CP Optimizer có `forbid_extent(interval, F)` với hàm bậc thang —
khai báo thẳng lịch của tài nguyên, **không sinh thêm biến quyết định nào**:

```python
fn = CpoStepFunction(); fn.set_value(0, horizon, 1)             # docplex.cp
for t in busy: fn.set_value(t, t + 1, 0)
mdl.add(mdl.forbid_extent(xt[i][tid], fn))
```

CP-SAT phải bung thành phép tuyển kèm **một biến bool cho mỗi cặp (khoá học, thời
điểm bận)**:

```python
after = model.new_bool_var(...)                                 # ortools
model.add(start[i] > t).only_enforce_if([lit, after])
model.add(end[i] <= t).only_enforce_if([lit, ~after])
```

Đây chính là "20+ năm chuyên biệt cho scheduling với interval variable" mà brief §1
nói tới, ở dạng cụ thể sờ được.